# Manipulação dos datasets da SIE

<img src="pipeline.jpg" width = 640>

## Básico - inicializações etc.

In [1]:
# Iniciando Spark
import findspark
findspark.init()

from pyspark.sql import SparkSession
spark = SparkSession.builder.master('local[4]').appName("tratamento_datasets_sie").getOrCreate()

# Carimbo data/hora
from _cabecalho_programas_notebook import *

# Renomeia todas e move pra raiz (d_painel)
def renomearRaiz():
    import os
    import shutil

    # Defina o caminho para a pasta principal
    pasta_principal = './d_painel'

    # Percorra todas as subpastas dentro da pasta principal
    for subpasta in os.listdir(pasta_principal):
        subpasta_path = os.path.join(pasta_principal, subpasta)
        
        # Verifique se a subpasta segue o padrão de título esperado
        if os.path.isdir(subpasta_path) and '_' in subpasta:
            
            # Obtenha o nome do arquivo CSV dentro da subpasta
            for arquivo in os.listdir(subpasta_path):
                if arquivo.endswith('.csv') or arquivo.endswith('.parquet'):
                    
                    # Construa o novo nome do arquivo removendo o prefixo de ano/mês/dia/hora/minuto/segundo
                    novo_nome = '_'.join(subpasta.split('_')[7:]) + os.path.splitext(arquivo)[1]
                    
                    # Obtenha o caminho completo para o arquivo atual
                    arquivo_atual = os.path.join(subpasta_path, arquivo)
                    
                    # Construa o caminho de destino para o arquivo na pasta raiz
                    caminho_destino = os.path.join(pasta_principal, novo_nome)
                    
                    # Renomeie o arquivo
                    novo_nome_completo = os.path.join(subpasta_path, novo_nome)
                    shutil.move(arquivo_atual, novo_nome_completo)
                    print(f'Arquivo renomeado: {arquivo_atual} -> {novo_nome_completo}')
                    
                    # Mova o arquivo renomeado para a pasta raiz
                    shutil.move(novo_nome_completo, caminho_destino)
                    print(f'Arquivo movido: {novo_nome_completo} -> {caminho_destino}')


## Etapa 1 - tratamento dos datasets brutos via Python e Pandas

In [ ]:
def get_deter(ano):
    import os
    import time
    import datetime
    import pandas as pd
    import numpy as np

    if not os.path.exists("./b_ref"):
        os.mkdir("./b_ref")

    if not os.path.exists("./b_ref/csv"):
        os.mkdir(f"./b_ref/csv")

    if not os.path.exists("./b_ref/parquet"):
        os.mkdir(f"./b_ref/parquet")

    print("Lendo o arquivo RAW")

    df = pd.read_csv(f"./a_raw/movimento/{ano}.csv", delimiter=';', encoding='cp1252', header = 1)

    df.replace("  ", "", regex=True, inplace=True)

    print("Renomeando colunas")
    if ano in [2020, 2022, 2023, 2024]:
        df.rename(columns={
            "Destino Seção": "des_localidade_nome",
            "Municipio Destino": "des_municipio_nome",
            "Transportadora": "empresa",
            "Km": "km",
            "Origem Seção": "ori_localidade_nome",
            "Municipio Origem": "ori_municipio_nome",
            "Passag.Estudante": "pax_jovem_gratis",
            "Passag.Comum": "pax_pagantes",
            "Passag.Total": "pax_total",
            "Linha/Ramal": "prefixo",
            "Seção": "sequencial",
            "Operacao": "servico_ambito"}, inplace=True)

        df.drop(["Itinerario", "Tempo", "Vl.Tarifa"], axis=1, inplace=True)

    elif ano == 2021:
        df["PASSAG.ESTUDANTE"].replace(" ", "", regex = True, inplace = True)
        df["PASSAG.COMUM"].replace(" ", "", regex = True, inplace = True)
        df["PASSAG.ESTUDANTE"].replace("", "0", regex = True, inplace = True)
        df["PASSAG.COMUM"].replace("", "0", regex = True, inplace = True)
        df["KMLEITO1"].replace(",", ".", regex = True, inplace = True)
        df["KMLEITO1"].replace("", "0.0", regex = True, inplace = True)
        df["KMLEITO2"].replace(",", ".", regex = True, inplace = True)
        df["KMLEITO2"].replace("", "0.0", regex = True, inplace = True)
        
        df = df.astype({"PASSAG.ESTUDANTE": int, "PASSAG.COMUM": int, "KMLEITO1" : float, "KMLEITO2" : float})
        
        df.rename(columns={
            "DESTINO DA SEÇÃO": "des_localidade_nome",
            "DESTINO DA LINHA": "des_municipio_nome",
            "TRANSPORTADORA": "empresa",
            "ORIGEM DA SEÇÃO": "ori_localidade_nome",
            "ORIGEM DA LINHA": "ori_municipio_nome",
            "PASSAG.ESTUDANTE": "pax_jovem_gratis",
            "PASSAG.COMUM": "pax_pagantes",
            "LINHA": "prefixo",
            "SEÇÃO": "sequencial",
            "CLASSIFICAO": "servico_ambito"}, inplace=True)

        df['pax_total'] = df['pax_jovem_gratis'] + df['pax_pagantes']
        
        df['km'] = df['KMLEITO1'] + df['KMLEITO2']
        
        df = df.astype({"km": str})

        df.drop(["ITINERARIO", "TEMPO(min)", "TARIFA", "PATAMAR",
                "KMLEITO1", "KMLEITO2", "RECEITA"], axis=1, inplace=True)

    else:
        df.rename(columns={
            "DESTINO": "des_localidade_nome",
            "MUNICIPIO DESTINO": "des_municipio_nome",
            "TRANSPORTADORA": "empresa",
            "KM": "km",
            "ORIGEM": "ori_localidade_nome",
            "MUNICIPIO ORIGEM": "ori_municipio_nome",
            "PASSAG.ESTUDANTE": "pax_jovem_gratis",
            "PASSAG.COMUM": "pax_pagantes",
            "PASSAG.TOTAL": "pax_total",
            "LINHA/RAMAL": "prefixo",
            "SECAO": "sequencial",
            "OPERACAO": "servico_ambito"}, inplace=True)

        df.drop(["ITINERARIO", "TEMPO"], axis=1, inplace=True)

    # cria colunas para uso futuro
    print("Criando novas colunas")

    df["ano"] = ano
    df["codigo"] = np.nan
    df["des_localidade_id"] = np.nan
    df["des_localidade_uf"] = np.nan
    df["des_municipio_id"] = np.nan
    df["empresa_cnpj"] = np.nan
    df["empresa_situacao"] = np.nan
    df["empresa_tipo"] = np.nan
    df["ida_idoso_desconto"] = np.nan
    df["ida_idoso_gratis"] = np.nan
    df["ida_jovem_desconto"] = np.nan
    df["ida_jovem_gratis"] = np.nan
    df["ida_pagantes"] = np.nan
    df["ida_passelivre"] = np.nan
    df["km_total"] = np.nan
    df["linha"] = np.nan
    df["linha_id"] = np.nan
    df["lugares_idas"] = np.nan
    df["lugares_voltas"] = np.nan
    df["mes"] = np.nan
    df["ori_localidade_id"] = np.nan
    df["ori_localidade_uf"] = np.nan
    df["ori_municipio_id"] = np.nan
    df["pax_gratis_descontos"] = np.nan
    df["pax_idoso_desconto"] = np.nan
    df["pax_idoso_gratis"] = np.nan
    df["pax_jovem_desconto"] = np.nan
    df["pax_passelivre"] = np.nan
    df["secao_id"] = np.nan
    df["servico_tipo"] = np.nan
    df["sisdap_fim"] = np.nan
    df["sisdap_inicio"] = np.nan
    df["viagem_idas"] = np.nan
    df["viagem_voltas"] = np.nan
    df["volta_idoso_desconto"] = np.nan
    df["volta_idoso_gratis"] = np.nan
    df["volta_jovem_desconto"] = np.nan
    df["volta_jovem_gratis"] = np.nan
    df["volta_pagantes"] = np.nan
    df["volta_passelivre"] = np.nan

    print("Limpando caracteres - str.strip()")
    df["ori_localidade_nome"] = df["ori_localidade_nome"].str.strip()
    df["des_localidade_nome"] = df["des_localidade_nome"].str.strip()
    df["ori_municipio_nome"] = df["ori_municipio_nome"].str.strip()
    df["des_municipio_nome"] = df["des_municipio_nome"].str.strip()

    print("Limpando colunas numéricas - prefixo, sequencial e km")
    df["prefixo"].replace(" ", "", regex=True, inplace=True)
    df["sequencial"].replace(" ", "", regex=True, inplace=True)
    df["km"].replace(" ", "", regex=True, inplace=True)
    df["pax_jovem_gratis"].replace(" ", "", regex=True, inplace=True)
    df["pax_pagantes"].replace(" ", "", regex=True, inplace=True)
    df["pax_total"].replace(" ", "", regex=True, inplace=True)

    print("Substituindo valores em branco por 0 - colunas pax_jovem_gratis, pax_pagantes e pax_total")
    df["pax_jovem_gratis"].replace("", '0', regex=True, inplace=True)
    df["pax_pagantes"].replace("", '0', regex=True, inplace=True)
    df["pax_total"].replace("", '0', regex=True, inplace=True)
    
    df = df.astype({"pax_jovem_gratis": int, "pax_pagantes": int, "pax_total" : int})

    print("Criando coluna ori_des_localidade_nome")
    # cria coluna com origen-destino localidade
    df["ori_des_localidade_nome"] = (
        df["ori_localidade_nome"] + "-" + df["des_localidade_nome"])

    print("Reindexando colunas por ordem alfabética")
    # reindexa colunas alfabeticamente
    df = df.reindex(sorted(df.columns), axis=1)

    print("Verificando o número de colunas do dataset")
    if len(df.columns) == 53:
        print(
            f"A quantidade de colunas ({len(df.columns)} colunas) do dataset está CORRETA.")
    else:
        print(
            f"A quantidade de colunas ({len(df.columns)} colunas) do dataset está INCORRETA.")

    df.to_csv(f"./b_ref/csv/{ano}.csv", sep=';', encoding='utf-8', index=False)
    print(f"Gravação no formato CSV efetuada para o dataset de {ano}")

    df.to_parquet(f"./b_ref/parquet/{ano}.parquet", compression='snappy', index=False)
    print(f"Gravação no formato Parquet efetuada para o dataset de {ano}")

    # end
    print(f"Sucesso para o dataset de {ano}\n")
    time.sleep(3)

In [ ]:
anos = range(1999, 2025, 1)

for ano in anos:
    get_deter(ano)

## Etapa 2 - tratamento via Spark

### Realiza join (união) entre os datasets limpos e os dicionários de linhas e seções

In [ ]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark.sql import SQLContext

# tb = spark.read.parquet("./b_ref/parquet")
# tb.createOrReplaceTempView("tb")

tb = spark.read.csv("./b_ref/csv/*.csv", sep = ";", encoding = "utf-8", header = True)
tb.createOrReplaceTempView("tb")

dicionario_secoes = spark.read.option("delimiter", ";").option("header", "true").option("inferSchema", "true").option("encoding", "utf-8").csv("_dicionarios/dicionario2023.csv")
dicionario_secoes.createOrReplaceTempView("dicionario_secoes")

dicionario_linhas = spark.read.option("delimiter", ";").option("header", "true").option("inferSchema", "true").option("encoding", "utf-8").csv("_dicionarios/dicionario2023.csv")
dicionario_linhas.createOrReplaceTempView("dicionario_linhas")

cidades = spark.read.csv("../BASE_CIDADES/dados/cidades.csv", sep = ";", header = True)
cidades.createOrReplaceTempView("cidades")

# verificação dos arquivos lidos

print(f"\
      tb = {tb.count()} linhas | {len(tb.columns)} colunas\n\
      dicionario_secoes = {dicionario_secoes.count()} linhas | {len(dicionario_secoes.columns)} colunas\n\
      dicionario_linhas = {dicionario_linhas.count()} linhas | {len(dicionario_linhas.columns)} colunas\
      ")

df_join = spark.sql("""

SELECT
-- base
tb.ano AS ano
,CAST(SUBSTRING_INDEX(tb.empresa, '-', 1) AS INTEGER) AS empresa_codigo
,tb.empresa AS empresa
,tb.servico_ambito AS modalidade
,CASE WHEN CAST(SUBSTRING_INDEX(tb.prefixo, '-', 1) AS INTEGER) >= 864 AND CAST(SUBSTRING_INDEX(tb.prefixo, '-', 1) AS INTEGER) <= 981 THEN 'Convênio ANTT' ELSE 'Intermunicipal catarinense' END AS interestadual
,tb.prefixo
,CAST(SUBSTRING_INDEX(tb.prefixo, '-', 1) AS INTEGER) AS prefixo_linha
,CAST(SUBSTRING_INDEX(tb.prefixo, '-', -1) AS INTEGER) AS prefixo_ramal
,CONCAT(tb.prefixo, ' - ', CONCAT(linha_ori.sig, '-', linha_des.sig)) AS linha
,CONCAT(tb.ano, '_', tb.prefixo, '_', CAST(SUBSTRING_INDEX(tb.empresa, '-', 1) AS INTEGER)) AS ano_prefixo_empresacod

-- linha_ori
,tb.ori_municipio_nome AS sie_linha_ori
,linha_ori.sig AS sig_linha_ori
,cidades_linha_ori.rgi_nome AS sig_linha_ori_rgi
,cidades_linha_ori.rgint_nome AS sig_linha_ori_rgint

-- linha_des
,tb.des_municipio_nome AS sie_linha_des
,linha_des.sig AS sig_linha_des
,cidades_linha_des.rgi_nome AS sig_linha_des_rgi
,cidades_linha_des.rgint_nome AS sig_linha_des_rgint

-- linha_ori_des
,CONCAT(linha_ori.sie, '-', linha_des.sie) AS sie_linha_ori_des
,CONCAT(linha_ori.sig, '-', linha_des.sig) AS sig_linha_ori_des
,CONCAT('LINESTRING(', cidades_linha_ori.mun_cidade_longitude, ' ', cidades_linha_ori.mun_cidade_latitude, ', ', cidades_linha_des.mun_cidade_longitude, ' ', cidades_linha_des.mun_cidade_latitude, ')') AS sig_linha_ori_des_wkt
,CASE WHEN cidades_linha_ori.rgi_nome = cidades_linha_des.rgi_nome THEN 'Intra' ELSE 'Inter' END AS sig_linha_rgi_escala
,CASE WHEN cidades_linha_ori.rgint_nome = cidades_linha_des.rgint_nome THEN 'Intra' ELSE 'Inter' END AS sig_linha_rgint_escala
,CASE WHEN linha_ori.sig = linha_des.sig THEN 'Intra' ELSE 'Inter' END AS sig_linha_municipal

-- secao
,CAST(tb.sequencial AS INTEGER) AS secao_sequencia
,CAST(REPLACE(tb.km, ',', '.') AS DOUBLE) AS km

-- secao_ori
,tb.ori_localidade_nome AS sie_secao_ori
,secao_ori.sig AS sig_secao_ori
,cidades_secao_ori.rgi_nome AS sig_secao_ori_rgi
,cidades_secao_ori.rgint_nome AS sig_secao_ori_rgint
,CONCAT('POINT(',cidades_secao_ori.mun_cidade_longitude, ' ', cidades_secao_ori.mun_cidade_latitude, ')') AS sig_secao_ori_wkt

-- secao_des
,tb.des_localidade_nome AS sie_secao_des
,secao_des.sig AS sig_secao_des
,cidades_secao_des.rgi_nome AS sig_secao_des_rgi
,cidades_secao_des.rgint_nome AS sig_secao_des_rgint
,CONCAT('POINT(',cidades_secao_des.mun_cidade_longitude, ' ', cidades_secao_des.mun_cidade_latitude, ')') AS sig_secao_des_wkt

-- secao ori_des
,CONCAT(secao_ori.sie, '-', secao_des.sie) AS sie_secao_ori_des
,CONCAT(secao_ori.sig, '-', secao_des.sig) AS sig_secao_ori_des
,CONCAT('LINESTRING(', cidades_secao_ori.mun_cidade_longitude, ' ', cidades_secao_ori.mun_cidade_latitude, ', ', cidades_secao_des.mun_cidade_longitude, ' ', cidades_secao_des.mun_cidade_latitude, ')') AS sig_secao_ori_des_wkt
,CASE WHEN cidades_secao_ori.rgi_nome = cidades_secao_des.rgi_nome THEN 'Intra' ELSE 'Inter' END AS sig_secao_rgi_escala
,CASE WHEN cidades_secao_ori.rgint_nome = cidades_secao_des.rgint_nome THEN 'Intra' ELSE 'Inter' END AS sig_secao_rgint_escala
,CASE WHEN secao_ori.sig = secao_des.sig THEN 'Intra' ELSE 'Inter' END AS sig_secao_municipal

-- secao_movimento
,CAST(tb.pax_total AS INTEGER) AS passageiros_total
,CAST(tb.pax_total * CAST(REPLACE(tb.km, ',', '.') AS DOUBLE) AS DOUBLE) AS pax_km

FROM tb

LEFT JOIN dicionario_secoes AS secao_ori ON tb.ori_localidade_nome = secao_ori.sie
LEFT JOIN dicionario_secoes AS secao_des ON tb.des_localidade_nome = secao_des.sie
LEFT JOIN dicionario_linhas AS linha_ori ON tb.ori_municipio_nome = linha_ori.sie
LEFT JOIN dicionario_linhas AS linha_des ON tb.des_municipio_nome = linha_des.sie

LEFT JOIN cidades AS cidades_linha_ori ON linha_ori.sig = cidades_linha_ori.mun_nomesigmai
LEFT JOIN cidades AS cidades_linha_des ON linha_des.sig = cidades_linha_des.mun_nomesigmai
LEFT JOIN cidades AS cidades_secao_ori ON secao_ori.sig = cidades_secao_ori.mun_nomesigmai
LEFT JOIN cidades AS cidades_secao_des ON secao_des.sig = cidades_secao_des.mun_nomesigmai

ORDER BY ano, empresa, prefixo

""")

# df_join = df_join.withColumn(
#       "empresa_codigo", split(df_join["empresa"], "-").getItem(0)).withColumn(
#             "prefixo_linha", split(df_join["prefixo"], "-").getItem(0)).withColumn(
#                   "prefixo_ramal", split(df_join["prefixo"], "-").getItem(1))

df_join.createOrReplaceTempView("df_join")

#### Testes df_join

In [ ]:
# spark.sql("""SELECT empresa, linha, passageiros_total FROM df_join WHERE ano = 2015
# """).toPandas().to_clipboard(index = False)
spark.sql("""SELECT ano AS `Ano`
    , SUM(CASE WHEN modalidade = 'URBANO' THEN passageiros_total ELSE 0 END) AS `Urbano`
    , SUM(CASE WHEN modalidade = 'RODOVIARIO' THEN passageiros_total ELSE 0 END) AS `Rodoviário`
    , SUM(passageiros_total) AS `Passageiros` FROM df_join WHERE sig_secao_municipal = 'Inter' GROUP BY ano ORDER BY ano ASC
""").show(900, False)

In [ ]:
spark.sql("SELECT * FROM df_join WHERE secao_sequencia IS NULL").show(1000, False)

In [ ]:
spark.sql("SELECT * FROM tb WHERE sequencial IS NULL").show(1000, False)

Check dicionário sem lacuna - secoes

Quantidade com problema

In [ ]:
spark.sql("""
SELECT COUNT(*) FROM (
SELECT
    ano,
    prefixo,
    sig_secao_ori_des,
    sie_secao_ori_des,
    sie_secao_ori,
    sie_secao_des
FROM
    df_join
WHERE
    sig_secao_ori IS NULL
    OR sig_secao_des IS NULL
    OR sig_secao_ori_des IS NULL)
""").show(100, False)

Quais são com problema

In [ ]:
spark.sql("""
SELECT
    ano,
    prefixo,
    sig_secao_ori_des,
    sie_secao_ori_des,
    sie_secao_ori,
    sie_secao_des
FROM
    df_join
WHERE
    sig_secao_ori IS NULL
    OR sig_secao_des IS NULL
    OR sig_secao_ori_des IS NULL
""").show(100, False)

Check dicionário sem lacuna - linhas

Quantidade com problema

In [ ]:
spark.sql("""
SELECT COUNT(*) FROM (
SELECT
    ano,
    prefixo,
    sig_linha_ori_des,
    sie_linha_ori_des,
    sie_linha_ori,
    sie_linha_des
FROM
    df_join
WHERE
    sig_linha_ori IS NULL
    OR sig_linha_des IS NULL
    OR sig_linha_ori_des IS NULL)
""").show(100, False)

Quais são com problema

In [ ]:
spark.sql("""
SELECT
    ano,
    prefixo,
    sig_linha_ori_des,
    sie_linha_ori_des,
    sie_linha_ori,
    sie_linha_des
FROM
    df_join
WHERE
    sig_linha_ori IS NULL
    OR sig_linha_des IS NULL
    OR sig_linha_ori_des IS NULL
""").show(100, False)

Visualização de teste do dataset com dicionário

In [ ]:
spark.sql("SELECT * FROM df_join WHERE prefixo LIKE '1020-%'").show(10, False)

### Gravação do dataset com dicionário

In [ ]:
print("Verificando integridade do dataframe com join (joined):")
print((df_join.count(), len(df_join.columns)))

# Parquet single file
df_join.repartition(1).write.option("header", "True").option("encoding", "utf-8").mode("overwrite").parquet("./c_ref_dicionarios/parquet/singlefile/")

# CSV singlefile
df_join.repartition(1).write.csv("./c_ref_dicionarios/csv/singlefile/", header = True, sep = ";", encoding = "utf-8")

## Etapa 3 - tratamentos para dashboard

### Lendo os parquets com join

In [ ]:
# df_spark = spark.read.option("schema", df_join.schema).parquet(f"./c_ref_dicionarios/parquet/singlefile/*.parquet")
df_spark = spark.read.parquet(f"./c_ref_dicionarios/parquet/singlefile/*.parquet")
df_spark.createOrReplaceTempView("data")

### 1/2 - Movimentação de passageiros, dashboard [CÓDIGO OK]

In [ ]:
movimentacao_full = spark.sql("SELECT * FROM data")

In [ ]:
spark.sql("""
SELECT
    ano AS `Ano`
    , SUM(CASE WHEN modalidade = 'URBANO' THEN passageiros_total ELSE 0 END) AS `Urbano`
    , SUM(CASE WHEN modalidade = 'RODOVIARIO' THEN passageiros_total ELSE 0 END) AS `Rodoviário`
    , SUM(passageiros_total) AS `Passageiros`
FROM
    data
WHERE
    ano BETWEEN 1999 AND 2023
GROUP BY
    ano
ORDER BY ano ASC
""").show(100, False)

#### Parquet single file

In [ ]:
movimentacao_full.repartition(1).write.option("header", "true").option("encoding", "utf-8").parquet(f"./d_painel/{get_time()}_1_trp_sc_movimentacao_full_parquet_single_file")

#### CSV single file

In [ ]:
movimentacao_full.repartition(1).write.option("header", "true").option("sep", ";").option("encoding", "utf-8").csv(f"./d_painel/{get_time()}_2_trp_sc_movimentacao_full_csv_single_file")

In [ ]:
# Parquet multiple file
# movimentacao_full.write.option("header", "True").option("encoding", "utf-8").parquet(f"./d_painel/{get_time()}_movimentacao_full_parquet_multiples_files/")

# CSV multiple file
# movimentacao_full.write.option("header", "true").option("sep", ",").option("encoding", "utf-8").csv(f"./d_painel/{get_time()}_movimentacao_full_csv_multiples_files")

# CSV single file e partitionby ano
# movimentacao_full.repartition(1).write.partitionBy("particao").option("header", "true").option("sep", ",").option("encoding", "utf-8").csv(f"./d_painel/{get_time()}_movimentacao_full_csv_particao_by_ano")

### 3 - Movimentação de passageiros por ano, cidade e modalidade [CÓDIGO OK]

In [ ]:
trp_sc_movimento_cidades_base = spark.sql("""
SELECT ano, cidade, modalidade, SUM(passageiros_total) AS passageiros FROM
(
SELECT ano, modalidade, sig_secao_ori AS cidade, CAST(passageiros_total AS INTEGER) FROM data WHERE sig_secao_ori != sig_secao_des
UNION ALL
SELECT ano, modalidade, sig_secao_des AS cidade, CAST(passageiros_total AS INTEGER) FROM data WHERE sig_secao_ori != sig_secao_des
UNION ALL
SELECT ano, modalidade, sig_secao_des AS cidade, CAST(passageiros_total AS INTEGER) FROM data WHERE sig_secao_ori = sig_secao_des
UNION ALL
SELECT ano, 'TOTAL' AS modalidade, sig_secao_ori AS cidade, CAST(passageiros_total AS INTEGER) FROM data WHERE sig_secao_ori != sig_secao_des
UNION ALL
SELECT ano, 'TOTAL' AS modalidade, sig_secao_des AS cidade, CAST(passageiros_total AS INTEGER) FROM data WHERE sig_secao_ori != sig_secao_des
UNION ALL
SELECT ano, 'TOTAL' AS modalidade, sig_secao_des AS cidade, CAST(passageiros_total AS INTEGER) FROM data WHERE sig_secao_ori = sig_secao_des
)
GROUP BY ano, cidade, modalidade
ORDER BY ano ASC, cidade ASC, modalidade ASC
""")
trp_sc_movimento_cidades_base.createOrReplaceTempView("trp_sc_movimento_cidades_base")

In [ ]:
trp_sc_movimento_cidades = spark.sql("""
SELECT
cidade AS municipio,
SUM(CASE WHEN ano = 1999 AND modalidade = 'RODOVIARIO' THEN passageiros ELSE 0 END) AS tot_p_r_1999, SUM(CASE WHEN ano = 1999 AND modalidade = 'URBANO' THEN passageiros ELSE 0 END) AS tot_p_u_1999, SUM(CASE WHEN ano = 1999 AND modalidade = 'TOTAL' THEN passageiros ELSE 0 END) AS tot_p_t_1999,
SUM(CASE WHEN ano = 2000 AND modalidade = 'RODOVIARIO' THEN passageiros ELSE 0 END) AS tot_p_r_2000, SUM(CASE WHEN ano = 2000 AND modalidade = 'URBANO' THEN passageiros ELSE 0 END) AS tot_p_u_2000, SUM(CASE WHEN ano = 2000 AND modalidade = 'TOTAL' THEN passageiros ELSE 0 END) AS tot_p_t_2000,
SUM(CASE WHEN ano = 2001 AND modalidade = 'RODOVIARIO' THEN passageiros ELSE 0 END) AS tot_p_r_2001, SUM(CASE WHEN ano = 2001 AND modalidade = 'URBANO' THEN passageiros ELSE 0 END) AS tot_p_u_2001, SUM(CASE WHEN ano = 2001 AND modalidade = 'TOTAL' THEN passageiros ELSE 0 END) AS tot_p_t_2001,
SUM(CASE WHEN ano = 2002 AND modalidade = 'RODOVIARIO' THEN passageiros ELSE 0 END) AS tot_p_r_2002, SUM(CASE WHEN ano = 2002 AND modalidade = 'URBANO' THEN passageiros ELSE 0 END) AS tot_p_u_2002, SUM(CASE WHEN ano = 2002 AND modalidade = 'TOTAL' THEN passageiros ELSE 0 END) AS tot_p_t_2002,
SUM(CASE WHEN ano = 2003 AND modalidade = 'RODOVIARIO' THEN passageiros ELSE 0 END) AS tot_p_r_2003, SUM(CASE WHEN ano = 2003 AND modalidade = 'URBANO' THEN passageiros ELSE 0 END) AS tot_p_u_2003, SUM(CASE WHEN ano = 2003 AND modalidade = 'TOTAL' THEN passageiros ELSE 0 END) AS tot_p_t_2003,
SUM(CASE WHEN ano = 2004 AND modalidade = 'RODOVIARIO' THEN passageiros ELSE 0 END) AS tot_p_r_2004, SUM(CASE WHEN ano = 2004 AND modalidade = 'URBANO' THEN passageiros ELSE 0 END) AS tot_p_u_2004, SUM(CASE WHEN ano = 2004 AND modalidade = 'TOTAL' THEN passageiros ELSE 0 END) AS tot_p_t_2004,
SUM(CASE WHEN ano = 2005 AND modalidade = 'RODOVIARIO' THEN passageiros ELSE 0 END) AS tot_p_r_2005, SUM(CASE WHEN ano = 2005 AND modalidade = 'URBANO' THEN passageiros ELSE 0 END) AS tot_p_u_2005, SUM(CASE WHEN ano = 2005 AND modalidade = 'TOTAL' THEN passageiros ELSE 0 END) AS tot_p_t_2005,
SUM(CASE WHEN ano = 2006 AND modalidade = 'RODOVIARIO' THEN passageiros ELSE 0 END) AS tot_p_r_2006, SUM(CASE WHEN ano = 2006 AND modalidade = 'URBANO' THEN passageiros ELSE 0 END) AS tot_p_u_2006, SUM(CASE WHEN ano = 2006 AND modalidade = 'TOTAL' THEN passageiros ELSE 0 END) AS tot_p_t_2006,
SUM(CASE WHEN ano = 2007 AND modalidade = 'RODOVIARIO' THEN passageiros ELSE 0 END) AS tot_p_r_2007, SUM(CASE WHEN ano = 2007 AND modalidade = 'URBANO' THEN passageiros ELSE 0 END) AS tot_p_u_2007, SUM(CASE WHEN ano = 2007 AND modalidade = 'TOTAL' THEN passageiros ELSE 0 END) AS tot_p_t_2007,
SUM(CASE WHEN ano = 2008 AND modalidade = 'RODOVIARIO' THEN passageiros ELSE 0 END) AS tot_p_r_2008, SUM(CASE WHEN ano = 2008 AND modalidade = 'URBANO' THEN passageiros ELSE 0 END) AS tot_p_u_2008, SUM(CASE WHEN ano = 2008 AND modalidade = 'TOTAL' THEN passageiros ELSE 0 END) AS tot_p_t_2008,
SUM(CASE WHEN ano = 2009 AND modalidade = 'RODOVIARIO' THEN passageiros ELSE 0 END) AS tot_p_r_2009, SUM(CASE WHEN ano = 2009 AND modalidade = 'URBANO' THEN passageiros ELSE 0 END) AS tot_p_u_2009, SUM(CASE WHEN ano = 2009 AND modalidade = 'TOTAL' THEN passageiros ELSE 0 END) AS tot_p_t_2009,
SUM(CASE WHEN ano = 2010 AND modalidade = 'RODOVIARIO' THEN passageiros ELSE 0 END) AS tot_p_r_2010, SUM(CASE WHEN ano = 2010 AND modalidade = 'URBANO' THEN passageiros ELSE 0 END) AS tot_p_u_2010, SUM(CASE WHEN ano = 2010 AND modalidade = 'TOTAL' THEN passageiros ELSE 0 END) AS tot_p_t_2010,
SUM(CASE WHEN ano = 2011 AND modalidade = 'RODOVIARIO' THEN passageiros ELSE 0 END) AS tot_p_r_2011, SUM(CASE WHEN ano = 2011 AND modalidade = 'URBANO' THEN passageiros ELSE 0 END) AS tot_p_u_2011, SUM(CASE WHEN ano = 2011 AND modalidade = 'TOTAL' THEN passageiros ELSE 0 END) AS tot_p_t_2011,
SUM(CASE WHEN ano = 2012 AND modalidade = 'RODOVIARIO' THEN passageiros ELSE 0 END) AS tot_p_r_2012, SUM(CASE WHEN ano = 2012 AND modalidade = 'URBANO' THEN passageiros ELSE 0 END) AS tot_p_u_2012, SUM(CASE WHEN ano = 2012 AND modalidade = 'TOTAL' THEN passageiros ELSE 0 END) AS tot_p_t_2012,
SUM(CASE WHEN ano = 2013 AND modalidade = 'RODOVIARIO' THEN passageiros ELSE 0 END) AS tot_p_r_2013, SUM(CASE WHEN ano = 2013 AND modalidade = 'URBANO' THEN passageiros ELSE 0 END) AS tot_p_u_2013, SUM(CASE WHEN ano = 2013 AND modalidade = 'TOTAL' THEN passageiros ELSE 0 END) AS tot_p_t_2013,
SUM(CASE WHEN ano = 2014 AND modalidade = 'RODOVIARIO' THEN passageiros ELSE 0 END) AS tot_p_r_2014, SUM(CASE WHEN ano = 2014 AND modalidade = 'URBANO' THEN passageiros ELSE 0 END) AS tot_p_u_2014, SUM(CASE WHEN ano = 2014 AND modalidade = 'TOTAL' THEN passageiros ELSE 0 END) AS tot_p_t_2014,
SUM(CASE WHEN ano = 2015 AND modalidade = 'RODOVIARIO' THEN passageiros ELSE 0 END) AS tot_p_r_2015, SUM(CASE WHEN ano = 2015 AND modalidade = 'URBANO' THEN passageiros ELSE 0 END) AS tot_p_u_2015, SUM(CASE WHEN ano = 2015 AND modalidade = 'TOTAL' THEN passageiros ELSE 0 END) AS tot_p_t_2015,
SUM(CASE WHEN ano = 2016 AND modalidade = 'RODOVIARIO' THEN passageiros ELSE 0 END) AS tot_p_r_2016, SUM(CASE WHEN ano = 2016 AND modalidade = 'URBANO' THEN passageiros ELSE 0 END) AS tot_p_u_2016, SUM(CASE WHEN ano = 2016 AND modalidade = 'TOTAL' THEN passageiros ELSE 0 END) AS tot_p_t_2016,
SUM(CASE WHEN ano = 2017 AND modalidade = 'RODOVIARIO' THEN passageiros ELSE 0 END) AS tot_p_r_2017, SUM(CASE WHEN ano = 2017 AND modalidade = 'URBANO' THEN passageiros ELSE 0 END) AS tot_p_u_2017, SUM(CASE WHEN ano = 2017 AND modalidade = 'TOTAL' THEN passageiros ELSE 0 END) AS tot_p_t_2017,
SUM(CASE WHEN ano = 2018 AND modalidade = 'RODOVIARIO' THEN passageiros ELSE 0 END) AS tot_p_r_2018, SUM(CASE WHEN ano = 2018 AND modalidade = 'URBANO' THEN passageiros ELSE 0 END) AS tot_p_u_2018, SUM(CASE WHEN ano = 2018 AND modalidade = 'TOTAL' THEN passageiros ELSE 0 END) AS tot_p_t_2018,
SUM(CASE WHEN ano = 2019 AND modalidade = 'RODOVIARIO' THEN passageiros ELSE 0 END) AS tot_p_r_2019, SUM(CASE WHEN ano = 2019 AND modalidade = 'URBANO' THEN passageiros ELSE 0 END) AS tot_p_u_2019, SUM(CASE WHEN ano = 2019 AND modalidade = 'TOTAL' THEN passageiros ELSE 0 END) AS tot_p_t_2019,
SUM(CASE WHEN ano = 2020 AND modalidade = 'RODOVIARIO' THEN passageiros ELSE 0 END) AS tot_p_r_2020, SUM(CASE WHEN ano = 2020 AND modalidade = 'URBANO' THEN passageiros ELSE 0 END) AS tot_p_u_2020, SUM(CASE WHEN ano = 2020 AND modalidade = 'TOTAL' THEN passageiros ELSE 0 END) AS tot_p_t_2020,
SUM(CASE WHEN ano = 2021 AND modalidade = 'RODOVIARIO' THEN passageiros ELSE 0 END) AS tot_p_r_2021, SUM(CASE WHEN ano = 2021 AND modalidade = 'URBANO' THEN passageiros ELSE 0 END) AS tot_p_u_2021, SUM(CASE WHEN ano = 2021 AND modalidade = 'TOTAL' THEN passageiros ELSE 0 END) AS tot_p_t_2021,
SUM(CASE WHEN ano = 2022 AND modalidade = 'RODOVIARIO' THEN passageiros ELSE 0 END) AS tot_p_r_2022, SUM(CASE WHEN ano = 2022 AND modalidade = 'URBANO' THEN passageiros ELSE 0 END) AS tot_p_u_2022, SUM(CASE WHEN ano = 2022 AND modalidade = 'TOTAL' THEN passageiros ELSE 0 END) AS tot_p_t_2022,
SUM(CASE WHEN ano = 2023 AND modalidade = 'RODOVIARIO' THEN passageiros ELSE 0 END) AS tot_p_r_2023, SUM(CASE WHEN ano = 2023 AND modalidade = 'URBANO' THEN passageiros ELSE 0 END) AS tot_p_u_2023, SUM(CASE WHEN ano = 2023 AND modalidade = 'TOTAL' THEN passageiros ELSE 0 END) AS tot_p_t_2023,
SUM(CASE WHEN ano = 2024 AND modalidade = 'RODOVIARIO' THEN passageiros ELSE 0 END) AS tot_p_r_2024, SUM(CASE WHEN ano = 2024 AND modalidade = 'URBANO' THEN passageiros ELSE 0 END) AS tot_p_u_2024, SUM(CASE WHEN ano = 2024 AND modalidade = 'TOTAL' THEN passageiros ELSE 0 END) AS tot_p_t_2024
FROM trp_sc_movimento_cidades_base
GROUP BY cidade
ORDER BY municipio ASC
""")
# trp_sc_movimento_cidades.show()
trp_sc_movimento_cidades.repartition(1).write.option("header", "true").option("sep", ";").option("encoding", "utf-8").csv(f"./d_painel/{get_time()}_3_trp_sc_movimento_cidades_pivot_tudo/")

### 4 - Movimentação de passageiros por ano, cidade e modalidade SEM PIVOT [CÓDIGO OK]

In [ ]:
trp_sc_movimento_cidades_notpivot = spark.sql("""
SELECT ano, cidade, modalidade, SUM(passageiros_total) FROM
(
SELECT ano, modalidade, sig_secao_ori AS cidade, CAST(passageiros_total AS INTEGER) FROM data WHERE sig_secao_ori != sig_secao_des
UNION ALL
SELECT ano, modalidade, sig_secao_des AS cidade, CAST(passageiros_total AS INTEGER) FROM data WHERE sig_secao_ori != sig_secao_des
UNION ALL
SELECT ano, modalidade, sig_secao_des AS cidade, CAST(passageiros_total AS INTEGER) FROM data WHERE sig_secao_ori = sig_secao_des
UNION ALL
SELECT ano, 'TOTAL' AS modalidade, sig_secao_ori AS cidade, CAST(passageiros_total AS INTEGER) FROM data WHERE sig_secao_ori != sig_secao_des
UNION ALL
SELECT ano, 'TOTAL' AS modalidade, sig_secao_des AS cidade, CAST(passageiros_total AS INTEGER) FROM data WHERE sig_secao_ori != sig_secao_des
UNION ALL
SELECT ano, 'TOTAL' AS modalidade, sig_secao_des AS cidade, CAST(passageiros_total AS INTEGER) FROM data WHERE sig_secao_ori = sig_secao_des
)
GROUP BY ano, cidade, modalidade
ORDER BY cidade ASC, ano ASC, modalidade ASC
""").repartition(1).write.option("header", "true").option("sep", ";").option("encoding", "utf-8").csv(f"./d_painel/{get_time()}_4_trp_sc_movimento_cidades_notpivot/")

In [ ]:
# filtra ANTT e fretamento
# trp_sc_movimento_cidades_notpivot = spark.sql("""
SELECT ano, cidade, modalidade, SUM(passageiros_total) FROM
(
SELECT ano, modalidade, sig_secao_ori AS cidade, CAST(passageiros_total AS INTEGER) FROM data WHERE sig_secao_ori != sig_secao_des AND prefixo NOT IN ('864-0', '865-0', '867-0', '868-0', '869-0', '870-0', '871-0', '872-0', '873-0', '874-0', '875-0', '876-0', '877-0', '878-0', '879-0', '880-0', '881-0', '882-0', '883-0', '884-0', '885-0', '886-0', '887-0', '888-0', '889-0', '890-0', '891-0', '892-0', '893-0', '894-0', '895-0', '896-0', '897-0', '898-0', '899-0', '900-0', '901-0', '902-0', '903-0', '904-0', '905-0', '906-0', '907-0', '908-0', '909-0', '910-0', '911-0', '912-0', '913-0', '914-0', '915-0', '916-0', '917-0', '918-0', '919-0', '920-0', '921-0', '922-0', '923-0', '924-0', '925-0', '926-0', '927-0', '928-0', '929-0', '930-0', '931-0', '932-0', '933-0', '934-0', '935-0', '936-0', '937-0', '938-0', '939-0', '940-0', '941-0', '942-0', '943-0', '944-0', '945-0', '946-0', '947-0', '948-0', '949-0', '950-0', '951-0', '952-0', '953-0', '954-0', '955-0', '956-0', '957-0', '958-0', '959-0', '960-0', '961-0', '962-0', '963-0', '964-0', '965-0', '966-0', '967-0', '968-0', '969-0', '970-0', '971-0', '972-0', '976-0', '977-0', '979-0', '980-0', '981-0') AND empresa_codigo NOT IN ('1462','1457','1737','1310','1273','2018','1741','1789')
UNION ALL
SELECT ano, modalidade, sig_secao_des AS cidade, CAST(passageiros_total AS INTEGER) FROM data WHERE sig_secao_ori != sig_secao_des AND prefixo NOT IN ('864-0', '865-0', '867-0', '868-0', '869-0', '870-0', '871-0', '872-0', '873-0', '874-0', '875-0', '876-0', '877-0', '878-0', '879-0', '880-0', '881-0', '882-0', '883-0', '884-0', '885-0', '886-0', '887-0', '888-0', '889-0', '890-0', '891-0', '892-0', '893-0', '894-0', '895-0', '896-0', '897-0', '898-0', '899-0', '900-0', '901-0', '902-0', '903-0', '904-0', '905-0', '906-0', '907-0', '908-0', '909-0', '910-0', '911-0', '912-0', '913-0', '914-0', '915-0', '916-0', '917-0', '918-0', '919-0', '920-0', '921-0', '922-0', '923-0', '924-0', '925-0', '926-0', '927-0', '928-0', '929-0', '930-0', '931-0', '932-0', '933-0', '934-0', '935-0', '936-0', '937-0', '938-0', '939-0', '940-0', '941-0', '942-0', '943-0', '944-0', '945-0', '946-0', '947-0', '948-0', '949-0', '950-0', '951-0', '952-0', '953-0', '954-0', '955-0', '956-0', '957-0', '958-0', '959-0', '960-0', '961-0', '962-0', '963-0', '964-0', '965-0', '966-0', '967-0', '968-0', '969-0', '970-0', '971-0', '972-0', '976-0', '977-0', '979-0', '980-0', '981-0') AND empresa_codigo NOT IN ('1462','1457','1737','1310','1273','2018','1741','1789')
UNION ALL
SELECT ano, modalidade, sig_secao_des AS cidade, CAST(passageiros_total AS INTEGER) FROM data WHERE sig_secao_ori = sig_secao_des AND prefixo NOT IN ('864-0', '865-0', '867-0', '868-0', '869-0', '870-0', '871-0', '872-0', '873-0', '874-0', '875-0', '876-0', '877-0', '878-0', '879-0', '880-0', '881-0', '882-0', '883-0', '884-0', '885-0', '886-0', '887-0', '888-0', '889-0', '890-0', '891-0', '892-0', '893-0', '894-0', '895-0', '896-0', '897-0', '898-0', '899-0', '900-0', '901-0', '902-0', '903-0', '904-0', '905-0', '906-0', '907-0', '908-0', '909-0', '910-0', '911-0', '912-0', '913-0', '914-0', '915-0', '916-0', '917-0', '918-0', '919-0', '920-0', '921-0', '922-0', '923-0', '924-0', '925-0', '926-0', '927-0', '928-0', '929-0', '930-0', '931-0', '932-0', '933-0', '934-0', '935-0', '936-0', '937-0', '938-0', '939-0', '940-0', '941-0', '942-0', '943-0', '944-0', '945-0', '946-0', '947-0', '948-0', '949-0', '950-0', '951-0', '952-0', '953-0', '954-0', '955-0', '956-0', '957-0', '958-0', '959-0', '960-0', '961-0', '962-0', '963-0', '964-0', '965-0', '966-0', '967-0', '968-0', '969-0', '970-0', '971-0', '972-0', '976-0', '977-0', '979-0', '980-0', '981-0') AND empresa_codigo NOT IN ('1462','1457','1737','1310','1273','2018','1741','1789')
UNION ALL
SELECT ano, 'TOTAL' AS modalidade, sig_secao_ori AS cidade, CAST(passageiros_total AS INTEGER) FROM data WHERE sig_secao_ori != sig_secao_des AND prefixo NOT IN ('864-0', '865-0', '867-0', '868-0', '869-0', '870-0', '871-0', '872-0', '873-0', '874-0', '875-0', '876-0', '877-0', '878-0', '879-0', '880-0', '881-0', '882-0', '883-0', '884-0', '885-0', '886-0', '887-0', '888-0', '889-0', '890-0', '891-0', '892-0', '893-0', '894-0', '895-0', '896-0', '897-0', '898-0', '899-0', '900-0', '901-0', '902-0', '903-0', '904-0', '905-0', '906-0', '907-0', '908-0', '909-0', '910-0', '911-0', '912-0', '913-0', '914-0', '915-0', '916-0', '917-0', '918-0', '919-0', '920-0', '921-0', '922-0', '923-0', '924-0', '925-0', '926-0', '927-0', '928-0', '929-0', '930-0', '931-0', '932-0', '933-0', '934-0', '935-0', '936-0', '937-0', '938-0', '939-0', '940-0', '941-0', '942-0', '943-0', '944-0', '945-0', '946-0', '947-0', '948-0', '949-0', '950-0', '951-0', '952-0', '953-0', '954-0', '955-0', '956-0', '957-0', '958-0', '959-0', '960-0', '961-0', '962-0', '963-0', '964-0', '965-0', '966-0', '967-0', '968-0', '969-0', '970-0', '971-0', '972-0', '976-0', '977-0', '979-0', '980-0', '981-0') AND empresa_codigo NOT IN ('1462','1457','1737','1310','1273','2018','1741','1789')
UNION ALL
SELECT ano, 'TOTAL' AS modalidade, sig_secao_des AS cidade, CAST(passageiros_total AS INTEGER) FROM data WHERE sig_secao_ori != sig_secao_des AND prefixo NOT IN ('864-0', '865-0', '867-0', '868-0', '869-0', '870-0', '871-0', '872-0', '873-0', '874-0', '875-0', '876-0', '877-0', '878-0', '879-0', '880-0', '881-0', '882-0', '883-0', '884-0', '885-0', '886-0', '887-0', '888-0', '889-0', '890-0', '891-0', '892-0', '893-0', '894-0', '895-0', '896-0', '897-0', '898-0', '899-0', '900-0', '901-0', '902-0', '903-0', '904-0', '905-0', '906-0', '907-0', '908-0', '909-0', '910-0', '911-0', '912-0', '913-0', '914-0', '915-0', '916-0', '917-0', '918-0', '919-0', '920-0', '921-0', '922-0', '923-0', '924-0', '925-0', '926-0', '927-0', '928-0', '929-0', '930-0', '931-0', '932-0', '933-0', '934-0', '935-0', '936-0', '937-0', '938-0', '939-0', '940-0', '941-0', '942-0', '943-0', '944-0', '945-0', '946-0', '947-0', '948-0', '949-0', '950-0', '951-0', '952-0', '953-0', '954-0', '955-0', '956-0', '957-0', '958-0', '959-0', '960-0', '961-0', '962-0', '963-0', '964-0', '965-0', '966-0', '967-0', '968-0', '969-0', '970-0', '971-0', '972-0', '976-0', '977-0', '979-0', '980-0', '981-0') AND empresa_codigo NOT IN ('1462','1457','1737','1310','1273','2018','1741','1789')
UNION ALL
SELECT ano, 'TOTAL' AS modalidade, sig_secao_des AS cidade, CAST(passageiros_total AS INTEGER) FROM data WHERE sig_secao_ori = sig_secao_des AND prefixo NOT IN ('864-0', '865-0', '867-0', '868-0', '869-0', '870-0', '871-0', '872-0', '873-0', '874-0', '875-0', '876-0', '877-0', '878-0', '879-0', '880-0', '881-0', '882-0', '883-0', '884-0', '885-0', '886-0', '887-0', '888-0', '889-0', '890-0', '891-0', '892-0', '893-0', '894-0', '895-0', '896-0', '897-0', '898-0', '899-0', '900-0', '901-0', '902-0', '903-0', '904-0', '905-0', '906-0', '907-0', '908-0', '909-0', '910-0', '911-0', '912-0', '913-0', '914-0', '915-0', '916-0', '917-0', '918-0', '919-0', '920-0', '921-0', '922-0', '923-0', '924-0', '925-0', '926-0', '927-0', '928-0', '929-0', '930-0', '931-0', '932-0', '933-0', '934-0', '935-0', '936-0', '937-0', '938-0', '939-0', '940-0', '941-0', '942-0', '943-0', '944-0', '945-0', '946-0', '947-0', '948-0', '949-0', '950-0', '951-0', '952-0', '953-0', '954-0', '955-0', '956-0', '957-0', '958-0', '959-0', '960-0', '961-0', '962-0', '963-0', '964-0', '965-0', '966-0', '967-0', '968-0', '969-0', '970-0', '971-0', '972-0', '976-0', '977-0', '979-0', '980-0', '981-0') AND empresa_codigo NOT IN ('1462','1457','1737','1310','1273','2018','1741','1789')
)
GROUP BY ano, cidade, modalidade
ORDER BY cidade ASC, ano ASC, modalidade ASC
""").repartition(1).write.option("header", "true").option("sep", ";").option("encoding", "utf-8").csv(f"./d_painel/{get_time()}_4_trp_sc_movimento_cidades_notpivot/")

### 5 - Linhas operadas por cidade [CÓDIGO OK]

In [ ]:
"""
************************************************
Próximas modificações:
- incorporar mudanças CÓDIGO EMPRESA, LINHA RAMAL, LINHA LINHA, EMPRESA INT etc.
************************************************
************ Legado em 2023-04-17 **************
************************************************
from pyspark.sql.functions import countDistinct
df_5 = spark.sql("SELECT ano, modalidade, sig_secao_ori AS cidade, empresa, linha AS linha_SIG, CONCAT(prefixo, ' - ', sie_linha_ori, ' - ', sie_linha_des) AS linha_SIE FROM data WHERE passageiros_total > 0")
df_6 = spark.sql("SELECT ano, modalidade, sig_secao_des AS cidade, empresa, linha AS linha_SIG, CONCAT(prefixo, ' - ', sie_linha_ori, ' - ', sie_linha_des) AS linha_SIE FROM data WHERE passageiros_total > 0")
df_7 = df_5.union(df_6)
df_8 = df_7.dropDuplicates()
df_9 = df_8.orderBy("ano", "cidade", "modalidade", "empresa", "linha_SIG")
************************************************
"""
linhas_por_cidade = spark.sql("""
SELECT DISTINCT(*) FROM (
SELECT
    ano,
    modalidade,
    sig_secao_ori AS cidade,
    empresa,
    prefixo,
    CAST(LEFT(prefixo, LENGTH(prefixo) - 2) AS INTEGER) AS prefixo_linha,
    RIGHT(prefixo, 1) AS prefixo_ramal,
    CONCAT(prefixo, ' - ', sig_linha_ori, ' - ', sig_linha_des) AS linha_SIG,
    CONCAT(sig_linha_ori, ' - ', sig_linha_des) AS linha_SIG_ori_des,
    CONCAT(prefixo, ' - ', sie_linha_ori, ' - ', sie_linha_des) AS linha_SIE,
    CONCAT(sie_linha_ori, ' - ', sie_linha_des) AS linha_SIE_ori_des
    , CASE WHEN CAST(passageiros_total AS INTEGER) > 0 THEN 'Com movimento' ELSE 'Sem movimento' END AS movimento_ano
FROM
    data
    
UNION ALL

SELECT
    ano,
    modalidade,
    sig_secao_des AS cidade,
    empresa,
    prefixo,
    CAST(LEFT(prefixo, LENGTH(prefixo) - 2) AS INTEGER) AS prefixo_linha,
    RIGHT(prefixo, 1) AS prefixo_ramal,
    CONCAT(prefixo, ' - ', sig_linha_ori, ' - ', sig_linha_des) AS linha_SIG,
    CONCAT(sig_linha_ori, ' - ', sig_linha_des) AS linha_SIG_ori_des,
    CONCAT(prefixo, ' - ', sie_linha_ori, ' - ', sie_linha_des) AS linha_SIE,
    CONCAT(sie_linha_ori, ' - ', sie_linha_des) AS linha_SIE_ori_des
    , CASE WHEN CAST(passageiros_total AS INTEGER) > 0 THEN 'Com movimento' ELSE 'Sem movimento' END AS movimento_ano
FROM
    data
)
ORDER BY ano, cidade ASC, modalidade ASC, empresa ASC, prefixo_linha ASC, prefixo_ramal ASC
""")
# ver(linhas_por_cidade)
linhas_por_cidade.repartition(1).write.option("header", "true").option("sep", ";").option("encoding", "utf-8").csv(f"./d_painel/{get_time()}_5_trp_sc_linhas_por_cidade/")

### 6 - Prefixos SC [CÓDIGO OK]

In [ ]:
spark.read.csv("../SIE/_dicionarios/dicionario_empresas_alias.csv", sep = ";", header = True).createOrReplaceTempView("aliasempresas")

In [ ]:
# spark.sql("SELECT ano, modalidade, CAST(SUM(passageiros_total) AS INTEGEr) AS pax FROM data GROUP BY ano, modalidade ORDER BY ano ASC").show(900, False)

In [ ]:
"""
************************************************************
Tabela com: ano, empresa, linha e se registrou movimentação no ano em questão
************************************************************
"""
santa_catarina_prefixos_1999_2023 = spark.sql("""
SELECT DISTINCT(*) FROM
(
SELECT
ano,
data.empresa_codigo,
aliasempresas.empresa_alias,
prefixo,
prefixo_linha,
prefixo_ramal,
modalidade,
sig_linha_rgi_escala,
sig_linha_rgint_escala,
extlinha.extenmax AS extensao,
linha_qtde_secoes.qtde_secoes AS secoes,
sie_linha_ori_des AS linha_sie,
sig_linha_ori_des AS linha_ibge,
pax.pax_total AS passageiros,
paxkm.passageiros_km,
CASE WHEN CAST(pax.pax_total AS INTEGER) > 0 THEN 'Sim' ELSE 'Não' END AS `movimento_no_ano`
FROM
data
LEFT JOIN aliasempresas ON aliasempresas.empresa_codigo = data.empresa_codigo
LEFT JOIN (SELECT CONCAT(ano, '_', prefixo, '_', empresa_codigo, '_', sig_linha_ori_des) AS diferenciador, MAX(CAST(REPLACE(km, ',', '.') AS FLOAT)) AS extenmax FROM data GROUP by CONCAT(ano, '_', prefixo, '_', empresa_codigo, '_', sig_linha_ori_des)) AS extlinha ON extlinha.diferenciador = CONCAT(data.ano, '_', data.prefixo, '_', data.empresa_codigo, '_', data.sig_linha_ori_des)
LEFT JOIN (SELECT CONCAT(ano, '_', prefixo, '_', empresa_codigo, '_', sig_linha_ori_des) AS diferenciador, COUNT(*) AS qtde_secoes FROM data GROUP by CONCAT(ano, '_', prefixo, '_', empresa_codigo, '_', sig_linha_ori_des)) AS linha_qtde_secoes ON linha_qtde_secoes.diferenciador = CONCAT(data.ano, '_', data.prefixo, '_', data.empresa_codigo, '_', data.sig_linha_ori_des)
LEFT JOIN (SELECT CONCAT(ano, '_', prefixo, '_', empresa_codigo, '_', sig_linha_ori_des) AS diferenciador, CAST(SUM(passageiros_total) AS INTEGER) pax_total FROM data GROUP by CONCAT(ano, '_', prefixo, '_', empresa_codigo, '_', sig_linha_ori_des)) AS pax ON pax.diferenciador = CONCAT(data.ano, '_', data.prefixo, '_', data.empresa_codigo, '_', data.sig_linha_ori_des)
LEFT JOIN (SELECT CONCAT(ano, '_', prefixo, '_', empresa_codigo, '_', sig_linha_ori_des) AS diferenciador, CAST(SUM(pax_km) AS INTEGER) passageiros_km FROM data GROUP by CONCAT(ano, '_', prefixo, '_', empresa_codigo, '_', sig_linha_ori_des)) AS paxkm ON paxkm.diferenciador = CONCAT(data.ano, '_', data.prefixo, '_', data.empresa_codigo, '_', data.sig_linha_ori_des)
--GROUP BY ano, data.empresa_codigo, aliasempresas.empresa_alias, prefixo, prefixo_linha, prefixo_ramal, modalidade, sie_linha_ori_des, sig_linha_ori_des, secoes, extlinha.extenmax
)
ORDER BY ano ASC, CAST(empresa_codigo AS INTEGER) ASC, CAST(prefixo_linha AS INTEGER) ASC, CAST(prefixo_ramal AS INTEGER) ASC
""")
# santa_catarina_prefixos_1999_2023.show(50, False)
# santa_catarina_prefixos_1999_2023.toPandas().to_csv("./d_painel/santa_catarina_prefixos_1999_2023.csv", index = False, encoding = "utf-8", sep = ";")

In [ ]:
santa_catarina_prefixos_1999_2023.createOrReplaceTempView("trp_sc_prefixos_1999_2023")

In [ ]:
santa_catarina_prefixos_1999_2023.repartition(1).write.option("header", "true").option("sep", ";").option("encoding", "utf-8").csv(f"d_painel/{get_time()}_6_trp_sc_prefixos_1999_2023/")

#### 6.1 - Prefixos SC tipologia [CÓDIGO OK]

In [ ]:
# import pyperclip
trp_sc_prefixos_1999_2023_tipologias = spark.sql(f"""
SELECT ano_prefixo_empresacod, FIRST(tipologia_regiao) AS tipologia_regiao FROM (
SELECT
CONCAT(ano, '_', prefixo, '_', empresa_codigo) AS ano_prefixo_empresacod
,ano
,empresa_codigo
,empresa_alias
,prefixo
,prefixo_linha
,prefixo_ramal
,modalidade
,extensao
,secoes
,passageiros
,sig_linha_rgi_escala
,sig_linha_rgint_escala
,linha_sie
,linha_ibge
,CASE
    WHEN modalidade = 'URBANO' THEN 'Semiurbano'
    WHEN extensao <= 75.0 THEN 'Semiurbano'
    WHEN linha_ibge LIKE '%FLORIANÓPOLISSC%' THEN 'Troncal'
    WHEN sig_linha_rgint_escala = 'Inter' AND extensao BETWEEN 75.0 AND 250.0 THEN 'Inter-regional média'
    WHEN sig_linha_rgint_escala = 'Inter' AND extensao > 250.0 THEN 'Inter-regional longa'
    WHEN sig_linha_rgint_escala = 'Intra' THEN 'Regional'
    ELSE 'Não se aplica'
    END AS tipologia_regiao
    
FROM trp_sc_prefixos_1999_2023
--WHERE ano = 2000 AND passageiros > 0 AND passageiros IS NOT NULL
ORDER BY
CAST(ano AS INTEGER) ASC
,CAST(empresa_codigo AS INTEGER) ASC
,CAST(prefixo_linha AS INTEGER) ASC
,CAST(prefixo_ramal AS INTEGER) ASC
                ) GROUP BY ano_prefixo_empresacod
""")

# pyperclip.copy(str(tuple(temp.toPandas()["prefixo"].to_list())))

In [ ]:
# import pyperclip
trp_sc_prefixos_1999_2023_tipologias = spark.sql(f"""
SELECT ano_prefixo_empresacod, FIRST(tipologia_regiao) AS tipologia_regiao FROM (
SELECT
CONCAT(ano, '_', prefixo, '_', empresa_codigo) AS ano_prefixo_empresacod
,ano
,empresa_codigo
,empresa_alias
,prefixo
,prefixo_linha
,prefixo_ramal
,modalidade
,extensao
,secoes
,passageiros
,sig_linha_rgi_escala
,sig_linha_rgint_escala
,linha_sie
,linha_ibge
,CASE
    WHEN modalidade = 'URBANO' THEN 'Semiurbano'
    WHEN extensao <= 75.0 THEN 'Semiurbano'
    WHEN linha_ibge LIKE '%FLORIANÓPOLISSC%' THEN 'Troncal'
    WHEN extensao BETWEEN 75.0 AND 250.0 THEN 'Regional média'
    WHEN extensao > 250.0 THEN 'Regional longa'
    ELSE 'Não se aplica'
    END AS tipologia_regiao
    
FROM trp_sc_prefixos_1999_2023
--WHERE ano = 2000 AND passageiros > 0 AND passageiros IS NOT NULL
ORDER BY
CAST(ano AS INTEGER) ASC
,CAST(empresa_codigo AS INTEGER) ASC
,CAST(prefixo_linha AS INTEGER) ASC
,CAST(prefixo_ramal AS INTEGER) ASC
                ) GROUP BY ano_prefixo_empresacod
""")

# pyperclip.copy(str(tuple(temp.toPandas()["prefixo"].to_list())))

In [ ]:
trp_sc_prefixos_1999_2023_tipologias.show(100, False)

In [ ]:
trp_sc_prefixos_1999_2023_tipologias.toPandas().to_clipboard(index = False, sep = ";")

In [ ]:
trp_sc_prefixos_1999_2023_tipologias.select("ano_prefixo_empresacod", "tipologia_regiao").toPandas().to_clipboard(index = False, sep = ";")

### 7 - Prefixos convênio ANTT [CÓDIGO OK]

In [ ]:
"""
************************************************************
Tabela com empresa e linha para check convênio ANTT
************************************************************
"""
trp_sc_prefixos_convenio_antt = spark.sql("""
SELECT DISTINCT(*) FROM
(
SELECT
data.empresa_codigo,
aliasempresas.empresa_alias,
prefixo,
prefixo_linha,
prefixo_ramal,
modalidade,
sie_linha_ori_des AS linha_sie,
sig_linha_ori_des AS linha_ibge,
CASE WHEN CAST(prefixo_linha AS INTEGER) >= 864 AND CAST(prefixo_linha AS INTEGER) <= 981 THEN 'Convênio ANTT' ELSE 'Intermunicipal catarinense' END AS interestadual
FROM
data
LEFT JOIN aliasempresas
ON aliasempresas.empresa_codigo = data.empresa_codigo
)
ORDER BY CAST(prefixo_linha AS INTEGER) ASC, CAST(prefixo_ramal AS INTEGER) ASC
""")

trp_sc_prefixos_convenio_antt.repartition(1).write.option("header", "true").option("sep", ";").option("encoding", "utf-8").csv(f"d_painel/{get_time()}_7_trp_sc_prefixos_convenio_antt/")

### 8 - Movimentação de passageiros para QGIS  [CÓDIGO OK] | elaborado em 2023-01-18 14:00:00

In [ ]:
def run_transform_to_qgis():
    df_qgis = spark.sql("""
    SELECT ano, sig_secao_ori_des, origem, destino, SUM(passageiros_total) AS passageiros FROM
    (SELECT
    CAST(ano AS STRING) AS ano,
    CAST(modalidade AS STRING) AS modalidade,
    CAST(sig_secao_ori AS STRING) AS origem,
    CAST(sig_secao_des AS STRING) AS destino,
    CONCAT(sig_secao_ori, ' - ', sig_secao_des) AS sig_secao_ori_des,
    CAST(passageiros_total AS INTEGER) AS passageiros_total
    FROM data
    WHERE passageiros_total > 0 AND NOT sig_secao_ori = sig_secao_des AND NOT (LENGTH(sig_secao_ori) = 2 OR LENGTH(sig_secao_des) = 2) /* AND modalidade = 'RODOVIARIO' */
    UNION ALL
    SELECT
    CAST(ano AS STRING) AS ano,
    CAST(modalidade AS STRING) AS modalidade,
    CAST(sig_secao_des AS STRING) AS origem,
    CAST(sig_secao_ori AS STRING) AS destino,
    CONCAT(sig_secao_des, ' - ', sig_secao_ori) AS sig_secao_ori_des,
    CAST(passageiros_total AS INTEGER) AS passageiros_total
    FROM data
    WHERE passageiros_total > 0 AND NOT sig_secao_ori = sig_secao_des AND NOT (LENGTH(sig_secao_ori) = 2 OR LENGTH(sig_secao_des) = 2) /* AND modalidade = 'RODOVIARIO'*/)
    GROUP BY ano, sig_secao_ori_des, origem, destino
    ORDER BY ano ASC, sig_secao_ori_des ASC
    """).drop("sig_secao_ori_des").repartition(1).write.option("header", "true").option("sep", ";").option("encoding", "utf-8").csv(f"./d_painel/{get_time()}_8_trp_sc_fluxos_ori_des_qgis/")

run_transform_to_qgis()

### 9 - Viagens realizadas

In [ ]:
# from pyspark.sql.types import *
# df_viagens_schema = StructType([
#     StructField("Transportadora", StringType(), True),
#     StructField("Linha/Ramal", StringType(), True),
#     StructField("Municipio Origem", StringType(), True),
#     StructField("Municipio Destino", StringType(), True),
#     StructField("Operacao", StringType(), True),
#     StructField("Itinerario", StringType(), True),
#     StructField("Viagens Realizadas", IntegerType(), True),
#     StructField("Lugares Ofertados", IntegerType(), True)
# ])
# df_viagens = spark.read.format("csv").schema(df_viagens_schema).option("header", "True").option("skipRows", 1).option("sep", ";").option("encoding", "cp1252").csv("./a_raw/viagens")
# df_viagens = spark.read.format("csv").option("header", "True").option("skipRows", 1).option("sep", ";").option("encoding", "cp1252").load("./a_raw/viagens")

In [ ]:
df_viagens = spark.read.csv("./a_raw/viagens", sep = ";", encoding = "cp1252", header = True)
df_viagens.repartition(1).write.csv("./d_painel/teste2", sep = ";", encoding = "utf-8", header = True)

In [ ]:
df_viagens.show()

In [ ]:
import os
import pandas as pd
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
pasta = "./a_raw/viagens"
csv = [arquivo for arquivo in os.listdir(pasta) if arquivo.endswith(".csv")]
df_viagens = pd.DataFrame()
for arquivo in csv:
    path = os.path.join(pasta, arquivo)
    dados = pd.read_csv(path, skiprows = 0, sep = ";", encoding = "cp1252")
    dados["ano"] = str(path[35:39])
    df_viagens = df_viagens.append(dados, ignore_index = True)
    
for coluna in df_viagens.columns:
    df_viagens[coluna] = df_viagens[coluna].astype("str")
    
for coluna in df_viagens.columns:
    df_viagens[coluna] = df_viagens[coluna].str.strip()

df_viagens["Linha/Ramal"] = df_viagens["Linha/Ramal"].str.replace(" ", "")

df_viagens = df_viagens.rename(columns = {
    "Transportadora" : "empresa",
    "Linha/Ramal" : "linha",
    "Municipio Origem" : "linha_origem",
    "Municipio Destino" : "linha_destino",
    "Operacao" : "linha_operacao",
    "Itinerario" : "linha_itinerario",
    "Viagens Realizadas" : "viagens",
    "Lugares Ofertados" : "lugares_ofertados",
    "ano" : "ano"
})

df_viagens["linha_descricao"] = df_viagens["linha"] + " - " + df_viagens["linha_origem"] + "-" + df_viagens["linha_destino"]

df_viagens["empresa_cod"] = df_viagens["empresa"].str.split("-").str.get(0)

df_viagens["linha_prefixo"] = df_viagens["linha"].str.split("-").str.get(0)

df_viagens["linha_ramal"] = df_viagens["linha"].str.split("-").str.get(1)

df_viagens = df_viagens.reindex(sorted(df_viagens.columns), axis=1)

import os
pasta = f"./d_painel/{get_time()}_9_trp_sc_viagens_realizadas/"

if not os.path.exists(pasta):
        os.mkdir(pasta)

df_viagens.to_csv(f"{pasta}/viagens_trp_sc.csv", encoding = "utf-8", sep = ";", index = False)

### 10/11/12/13 - Análise da movimentação de passageiros com destaque para a km das seções

#### COM ANO PIVOTADO

In [ ]:
# ::::::::::::::::: COM DIFERENCIAÇÃO DA MODALIDADE ::::::::::::::::::::::::::::
df_analise_km_secoes_modalidades_anopivot = spark.sql("""
SELECT ano, modalidade, km, SUM(CAST(passageiros_total AS INTEGER)) AS passageiros FROM
(SELECT
    ano,
    CASE WHEN modalidade = 'RODOVIARIO' THEN 'Rodoviário' WHEN modalidade = 'URBANO' THEN 'Urbano' END AS modalidade,
    CASE
        WHEN CAST(REPLACE(km, ',', '.') AS FLOAT) >= 0 AND CAST(REPLACE(km, ',', '.') AS FLOAT) <= 10.0 THEN 'a) Até 10,0 km'
        WHEN CAST(REPLACE(km, ',', '.') AS FLOAT) > 10.0 AND CAST(REPLACE(km, ',', '.') AS FLOAT) <= 20.0 THEN 'b) 10,1 km - 20,0 km'
        WHEN CAST(REPLACE(km, ',', '.') AS FLOAT) > 20.0 AND CAST(REPLACE(km, ',', '.') AS FLOAT) <= 30.0 THEN 'c) 20,1 km - 30,0 km'
        WHEN CAST(REPLACE(km, ',', '.') AS FLOAT) > 30.0 AND CAST(REPLACE(km, ',', '.') AS FLOAT) <= 40.0 THEN 'd) 30,1 km - 40,0 km'
        WHEN CAST(REPLACE(km, ',', '.') AS FLOAT) > 40.0 AND CAST(REPLACE(km, ',', '.') AS FLOAT) <= 50.0 THEN 'e) 40,1 km - 50,0 km'
        WHEN CAST(REPLACE(km, ',', '.') AS FLOAT) > 50.0 AND CAST(REPLACE(km, ',', '.') AS FLOAT) <= 60.0 THEN 'f) 50,1 km - 60,0 km'
        WHEN CAST(REPLACE(km, ',', '.') AS FLOAT) > 60.0 AND CAST(REPLACE(km, ',', '.') AS FLOAT) <= 70.0 THEN 'g) 60,1 km - 70,0 km'
        WHEN CAST(REPLACE(km, ',', '.') AS FLOAT) > 70.0 AND CAST(REPLACE(km, ',', '.') AS FLOAT) <= 80.0 THEN 'h) 70,1 km - 80,0 km'
        WHEN CAST(REPLACE(km, ',', '.') AS FLOAT) > 80.0 AND CAST(REPLACE(km, ',', '.') AS FLOAT) <= 90.0 THEN 'i) 80,1 km - 90,0 km'
        WHEN CAST(REPLACE(km, ',', '.') AS FLOAT) > 90.0 AND CAST(REPLACE(km, ',', '.') AS FLOAT) <= 100.0 THEN 'j) 90,1 km - 100,0 km'
        WHEN CAST(REPLACE(km, ',', '.') AS FLOAT) > 100.0 AND CAST(REPLACE(km, ',', '.') AS FLOAT) <= 250.0 THEN 'k) 100,1 km - 250,0 km'
        WHEN CAST(REPLACE(km, ',', '.') AS FLOAT) > 250.0 AND CAST(REPLACE(km, ',', '.') AS FLOAT) <= 500.0 THEN 'l) 250,1 - 500,0 km'
        ELSE 'm) Acima de 500,0 km'
    END AS km,
    passageiros_total
FROM
    data
WHERE
    --modalidade IN ('RODOVIARIO') --AND
    --empresa LIKE '%REUNIDAS%' AND
    --km LIKE ',%' AND
    --passageiros_total > 0 AND
    --ano IN (2000, 2005, 2010, 2019, 2020, 2021, 2022)
) GROUP BY ano, modalidade, km
""").groupBy("modalidade", "km").pivot("ano").sum("passageiros").orderBy("modalidade", "km")
# df_analise_km_secoes_modalidades_anopivot.show(10, False)
# .groupBy("modalidade", "km").pivot("ano").sum("passageiros").orderBy("modalidade", "km").toPandas().to_clipboard(decimal = ",", index = False, sep = ";")
# ver(df_analise_km_secoes_modalidades_anopivot)

In [ ]:
# ::::::::::::::::::::::::::::: SEM ::::::::::::::::::::::::::::::::::::::::
# ::::::::::::::::: DIFERENCIAÇÃO DA MODALIDADE ::::::::::::::::::::::::::::
df_analise_km_secoes_anopivot = spark.sql("""
SELECT ano, 'Total' AS modalidade, km, SUM(CAST(passageiros_total AS INTEGER)) AS passageiros FROM
(SELECT
    ano,
    CASE
        WHEN CAST(REPLACE(km, ',', '.') AS FLOAT) >= 0 AND CAST(REPLACE(km, ',', '.') AS FLOAT) <= 10.0 THEN 'a) Até 10,0 km'
        WHEN CAST(REPLACE(km, ',', '.') AS FLOAT) > 10.0 AND CAST(REPLACE(km, ',', '.') AS FLOAT) <= 20.0 THEN 'b) 10,1 km - 20,0 km'
        WHEN CAST(REPLACE(km, ',', '.') AS FLOAT) > 20.0 AND CAST(REPLACE(km, ',', '.') AS FLOAT) <= 30.0 THEN 'c) 20,1 km - 30,0 km'
        WHEN CAST(REPLACE(km, ',', '.') AS FLOAT) > 30.0 AND CAST(REPLACE(km, ',', '.') AS FLOAT) <= 40.0 THEN 'd) 30,1 km - 40,0 km'
        WHEN CAST(REPLACE(km, ',', '.') AS FLOAT) > 40.0 AND CAST(REPLACE(km, ',', '.') AS FLOAT) <= 50.0 THEN 'e) 40,1 km - 50,0 km'
        WHEN CAST(REPLACE(km, ',', '.') AS FLOAT) > 50.0 AND CAST(REPLACE(km, ',', '.') AS FLOAT) <= 60.0 THEN 'f) 50,1 km - 60,0 km'
        WHEN CAST(REPLACE(km, ',', '.') AS FLOAT) > 60.0 AND CAST(REPLACE(km, ',', '.') AS FLOAT) <= 70.0 THEN 'g) 60,1 km - 70,0 km'
        WHEN CAST(REPLACE(km, ',', '.') AS FLOAT) > 70.0 AND CAST(REPLACE(km, ',', '.') AS FLOAT) <= 80.0 THEN 'h) 70,1 km - 80,0 km'
        WHEN CAST(REPLACE(km, ',', '.') AS FLOAT) > 80.0 AND CAST(REPLACE(km, ',', '.') AS FLOAT) <= 90.0 THEN 'i) 80,1 km - 90,0 km'
        WHEN CAST(REPLACE(km, ',', '.') AS FLOAT) > 90.0 AND CAST(REPLACE(km, ',', '.') AS FLOAT) <= 100.0 THEN 'j) 90,1 km - 100,0 km'
        WHEN CAST(REPLACE(km, ',', '.') AS FLOAT) > 100.0 AND CAST(REPLACE(km, ',', '.') AS FLOAT) <= 250.0 THEN 'k) 100,1 km - 250,0 km'
        WHEN CAST(REPLACE(km, ',', '.') AS FLOAT) > 250.0 AND CAST(REPLACE(km, ',', '.') AS FLOAT) <= 500.0 THEN 'l) 250,1 - 500,0 km'
        ELSE 'm) Acima de 500,0 km'
    END AS km,
    passageiros_total
FROM
    data
WHERE
    --modalidade IN ('RODOVIARIO') --AND
    --empresa LIKE '%REUNIDAS%' AND
    --km LIKE ',%' AND
    --passageiros_total > 0 AND
    --ano IN (2000, 2005, 2010, 2019, 2020, 2021, 2022)
) GROUP BY ano, km
""").groupBy("modalidade", "km").pivot("ano").sum("passageiros").orderBy("modalidade", "km")
# df_analise_km_secoes_anopivot.show(10, False)

#### Não pivotado

In [ ]:
# ::::::::::::::::: COM DIFERENCIAÇÃO DA MODALIDADE ::::::::::::::::::::::::::::
df_analise_km_secoes_modalidades = spark.sql("""
SELECT ano, modalidade, km, SUM(CAST(passageiros_total AS INTEGER)) AS passageiros FROM
(SELECT
    ano,
    CASE WHEN modalidade = 'RODOVIARIO' THEN 'Rodoviário' WHEN modalidade = 'URBANO' THEN 'Urbano' END AS modalidade,
    CASE
        WHEN CAST(REPLACE(km, ',', '.') AS FLOAT) >= 0 AND CAST(REPLACE(km, ',', '.') AS FLOAT) <= 10.0 THEN 'a) Até 10,0 km'
        WHEN CAST(REPLACE(km, ',', '.') AS FLOAT) > 10.0 AND CAST(REPLACE(km, ',', '.') AS FLOAT) <= 20.0 THEN 'b) 10,1 km - 20,0 km'
        WHEN CAST(REPLACE(km, ',', '.') AS FLOAT) > 20.0 AND CAST(REPLACE(km, ',', '.') AS FLOAT) <= 30.0 THEN 'c) 20,1 km - 30,0 km'
        WHEN CAST(REPLACE(km, ',', '.') AS FLOAT) > 30.0 AND CAST(REPLACE(km, ',', '.') AS FLOAT) <= 40.0 THEN 'd) 30,1 km - 40,0 km'
        WHEN CAST(REPLACE(km, ',', '.') AS FLOAT) > 40.0 AND CAST(REPLACE(km, ',', '.') AS FLOAT) <= 50.0 THEN 'e) 40,1 km - 50,0 km'
        WHEN CAST(REPLACE(km, ',', '.') AS FLOAT) > 50.0 AND CAST(REPLACE(km, ',', '.') AS FLOAT) <= 60.0 THEN 'f) 50,1 km - 60,0 km'
        WHEN CAST(REPLACE(km, ',', '.') AS FLOAT) > 60.0 AND CAST(REPLACE(km, ',', '.') AS FLOAT) <= 70.0 THEN 'g) 60,1 km - 70,0 km'
        WHEN CAST(REPLACE(km, ',', '.') AS FLOAT) > 70.0 AND CAST(REPLACE(km, ',', '.') AS FLOAT) <= 80.0 THEN 'h) 70,1 km - 80,0 km'
        WHEN CAST(REPLACE(km, ',', '.') AS FLOAT) > 80.0 AND CAST(REPLACE(km, ',', '.') AS FLOAT) <= 90.0 THEN 'i) 80,1 km - 90,0 km'
        WHEN CAST(REPLACE(km, ',', '.') AS FLOAT) > 90.0 AND CAST(REPLACE(km, ',', '.') AS FLOAT) <= 100.0 THEN 'j) 90,1 km - 100,0 km'
        WHEN CAST(REPLACE(km, ',', '.') AS FLOAT) > 100.0 AND CAST(REPLACE(km, ',', '.') AS FLOAT) <= 250.0 THEN 'k) 100,1 km - 250,0 km'
        WHEN CAST(REPLACE(km, ',', '.') AS FLOAT) > 250.0 AND CAST(REPLACE(km, ',', '.') AS FLOAT) <= 500.0 THEN 'l) 250,1 - 500,0 km'
        ELSE 'm) Acima de 500,0 km'
    END AS km,
    passageiros_total
FROM
    data
WHERE
    --modalidade IN ('RODOVIARIO') --AND
    --empresa LIKE '%REUNIDAS%' AND
    --km LIKE ',%' AND
    --passageiros_total > 0 AND
    --ano IN (2000, 2005, 2010, 2019, 2020, 2021, 2022)
) GROUP BY ano, modalidade, km
""").groupBy("ano", "modalidade", "km").sum("passageiros").orderBy("ano", "modalidade", "km")
# df_analise_km_secoes_modalidades.show(10, False)

In [ ]:
# ::::::::::::::::::::::::::::: SEM ::::::::::::::::::::::::::::::::::::::::
# ::::::::::::::::: DIFERENCIAÇÃO DA MODALIDADE ::::::::::::::::::::::::::::
df_analise_km_secoes = spark.sql("""
SELECT ano, 'Total' AS modalidade, km, SUM(CAST(passageiros_total AS INTEGER)) AS passageiros FROM
(SELECT
    ano,
    CASE
        WHEN CAST(REPLACE(km, ',', '.') AS FLOAT) >= 0 AND CAST(REPLACE(km, ',', '.') AS FLOAT) <= 10.0 THEN 'a) Até 10,0 km'
        WHEN CAST(REPLACE(km, ',', '.') AS FLOAT) > 10.0 AND CAST(REPLACE(km, ',', '.') AS FLOAT) <= 20.0 THEN 'b) 10,1 km - 20,0 km'
        WHEN CAST(REPLACE(km, ',', '.') AS FLOAT) > 20.0 AND CAST(REPLACE(km, ',', '.') AS FLOAT) <= 30.0 THEN 'c) 20,1 km - 30,0 km'
        WHEN CAST(REPLACE(km, ',', '.') AS FLOAT) > 30.0 AND CAST(REPLACE(km, ',', '.') AS FLOAT) <= 40.0 THEN 'd) 30,1 km - 40,0 km'
        WHEN CAST(REPLACE(km, ',', '.') AS FLOAT) > 40.0 AND CAST(REPLACE(km, ',', '.') AS FLOAT) <= 50.0 THEN 'e) 40,1 km - 50,0 km'
        WHEN CAST(REPLACE(km, ',', '.') AS FLOAT) > 50.0 AND CAST(REPLACE(km, ',', '.') AS FLOAT) <= 60.0 THEN 'f) 50,1 km - 60,0 km'
        WHEN CAST(REPLACE(km, ',', '.') AS FLOAT) > 60.0 AND CAST(REPLACE(km, ',', '.') AS FLOAT) <= 70.0 THEN 'g) 60,1 km - 70,0 km'
        WHEN CAST(REPLACE(km, ',', '.') AS FLOAT) > 70.0 AND CAST(REPLACE(km, ',', '.') AS FLOAT) <= 80.0 THEN 'h) 70,1 km - 80,0 km'
        WHEN CAST(REPLACE(km, ',', '.') AS FLOAT) > 80.0 AND CAST(REPLACE(km, ',', '.') AS FLOAT) <= 90.0 THEN 'i) 80,1 km - 90,0 km'
        WHEN CAST(REPLACE(km, ',', '.') AS FLOAT) > 90.0 AND CAST(REPLACE(km, ',', '.') AS FLOAT) <= 100.0 THEN 'j) 90,1 km - 100,0 km'
        WHEN CAST(REPLACE(km, ',', '.') AS FLOAT) > 100.0 AND CAST(REPLACE(km, ',', '.') AS FLOAT) <= 250.0 THEN 'k) 100,1 km - 250,0 km'
        WHEN CAST(REPLACE(km, ',', '.') AS FLOAT) > 250.0 AND CAST(REPLACE(km, ',', '.') AS FLOAT) <= 500.0 THEN 'l) 250,1 - 500,0 km'
        ELSE 'm) Acima de 500,0 km'
    END AS km,
    passageiros_total
FROM
    data
WHERE
    --modalidade IN ('RODOVIARIO') --AND
    --empresa LIKE '%REUNIDAS%' AND
    --km LIKE ',%' AND
    --passageiros_total > 0 AND
    --ano IN (2000, 2005, 2010, 2019, 2020, 2021, 2022)
) GROUP BY ano, km
""").groupBy("ano", "modalidade", "km").sum("passageiros").orderBy("ano", "modalidade", "km")
# df_analise_km_secoes.show(10, False)

In [ ]:
df_analise_km_secoes_modalidades_anopivot.repartition(1).write.csv(f"./d_painel/{get_time()}_10_trp_sc_analise_km_secoes_modalidades_anopivot/", header = True, sep = ";", encoding = "utf-8")
df_analise_km_secoes_anopivot.repartition(1).write.csv(f"./d_painel/{get_time()}_11_trp_sc_analise_km_secoes_anopivot/", header = True, sep = ";", encoding = "utf-8")
df_analise_km_secoes_modalidades.repartition(1).write.csv(f"./d_painel/{get_time()}_12_trp_sc_analise_km_secoes_modalidades/", header = True, sep = ";", encoding = "utf-8")
df_analise_km_secoes.repartition(1).write.csv(f"./d_painel/{get_time()}_13_trp_sc_analise_km_secoes/", header = True, sep = ";", encoding = "utf-8")

### 14/15 - linhas por extensão

Ext. linhas ord. por ext.

In [ ]:
spark.sql("""
SELECT 'Ext. linhas ord. por ext.' AS tabela, ano, empresa, REPLACE(LEFT(linha, LENGTH(linha) - 2), 'SC-', ' - ') AS linha, prefixo_linha, prefixo_ramal, MAX(kmnovo) AS extensao FROM
(
SELECT *, CAST(REPLACE(km, ',', '.') AS DECIMAL(4,1)) AS kmnovo FROM data
)
--WHERE ano = 2000 --AND prefixo_linha <= 300
GROUP BY ano, empresa, linha, prefixo_linha, prefixo_ramal
ORDER BY ano ASC, extensao DESC
""").repartition(1).write.csv(f"./d_painel/{get_time()}_14_trp_sc_linhas_classif_por_ext/", header = True, sep = ";", encoding = "utf-8")
# .show(10, False)
# .toPandas().to_clipboard(sep = ";", index = False)

Ext. linhas ord. por prefixo/ramal

In [ ]:
spark.sql("""
SELECT 'Ext. linhas ord. por prefixo/ramal' AS tabela, ano, empresa, REPLACE(LEFT(linha, LENGTH(linha) - 2), 'SC-', ' - ') AS linha, prefixo_linha, prefixo_ramal, MAX(kmnovo) AS extensao FROM
(
SELECT *, CAST(REPLACE(km, ',', '.') AS DECIMAL(4,1)) AS kmnovo FROM data
)
--WHERE ano = 2000
GROUP BY ano, empresa, linha, prefixo_linha, prefixo_ramal
ORDER BY ano ASC, CAST(prefixo_linha AS INTEGER) ASC, CAST(prefixo_ramal AS INTEGER) ASC
""").repartition(1).write.csv(f"./d_painel/{get_time()}_15_trp_sc_linhas_classif_por_pfx_ram/", header = True, sep = ";", encoding = "utf-8")
# .show(10, False)
# .toPandas().to_clipboard(sep = ";", index = False)

In [ ]:
# Renomeando e movendo os arquivos para a pasta d_painel raiz
renomearRaiz()

### 16 - Tabela síntese da dinâmica recente e organização espacial do TRP catarinense

In [2]:
df = spark.read.parquet("./d_painel/trp_sc_movimentacao_full_parquet_single_file.parquet")
df.createOrReplaceTempView("df")

spark.read.csv("../SIE/_dicionarios/dicionario_empresas_alias.csv", sep = ";", header = True).createOrReplaceTempView("aliasempresas")

cidades = spark.read.csv("../BASE_CIDADES/dados/cidades.csv", sep = ";", header = True)
cidades.createOrReplaceTempView("cidades")

cidades_pop_2022 = spark.read.csv("../BASE_CIDADES/dados/brasil_pop_2022_censo_20230628.csv", sep = ";", header = True)
cidades_pop_2022.createOrReplaceTempView("cidades_pop_2022")

brasil_regic_2018_hierarquia_geocodigos = spark.read.csv("../BASE_CIDADES/dados/brasil_regic_2018_hierarquia_geocodigos.csv", sep = ";", header = True)
brasil_regic_2018_hierarquia_geocodigos.createOrReplaceTempView("brasil_regic_2018_hierarquia_geocodigos")

br_dados_sintese_2010_2022 = spark.read.csv("../BASE_CIDADES/br_dados_sintese_2010_2022.csv", sep = ";", header = True)
br_dados_sintese_2010_2022.createOrReplaceTempView("br_dados_sintese_2010_2022")

In [ ]:
esquemaop = spark.read.parquet(r"D:\Geografia\0_DADOS_ORGANIZADOS\TRANSPORTE\ANTT\ANTT_pontos_linhas_esq_op\pontos_reunidos.parquet")
diciov3full = spark.read.csv(r"D:\Geografia\0_DADOS_ORGANIZADOS\TRANSPORTE\ANTT\ANTT_banco_de_dados_passageiros\DICIONARIO_ANTT_2023_INFRASA_v3.csv", sep = ";", encoding = "utf-8", header = True)
esquemaop.createOrReplaceTempView("esquemaop")
diciov3full.createOrReplaceTempView("diciov3full")

In [5]:
ano_definicao = 2010

df_trp_sc_tabela_sintese_org_espacial = spark.sql(f"""
SELECT
cidades.mun_nomesigmai
,cidades.geocodigo
,cidades.rgint_nome
,cidades.rgi_nome
,CAST(total_ligacoes.qtde AS INTEGER) AS qtde_linhas
,CAST(ligacoes_sao_ori_des.qtde AS INTEGER) AS qtde_linhas_terminais
--                                              ,CAST(antt.qtde AS INTEGER) AS qtde_linhas_interestaduais
,CAST(passageiros_1999.qtde AS INTEGER) AS passageiros_1999
,CAST(passageiros_2010.qtde AS INTEGER) AS passageiros_2010
,CAST(passageiros_2019.qtde AS INTEGER) AS passageiros_2019
,CAST(passageiros_2023.qtde AS INTEGER) AS passageiros_2023
,CAST(passageiros_1999_2023.qtde AS INTEGER) AS passageiros_1999_2023
,CASE WHEN CAST(passageiros_2010.qtde AS INTEGER) IS NULL THEN 0 ELSE CAST(passageiros_2010.qtde AS INTEGER) END - CASE WHEN CAST(passageiros_1999.qtde AS INTEGER) IS NULL THEN 0 ELSE CAST(passageiros_1999.qtde AS INTEGER) END AS dif_passageiros_2010_1999
,CASE WHEN CAST(passageiros_2019.qtde AS INTEGER) IS NULL THEN 0 ELSE CAST(passageiros_2019.qtde AS INTEGER) END - CASE WHEN CAST(passageiros_2010.qtde AS INTEGER) IS NULL THEN 0 ELSE CAST(passageiros_2010.qtde AS INTEGER) END AS dif_passageiros_2019_2010
,CASE WHEN CAST(passageiros_2019.qtde AS INTEGER) IS NULL THEN 0 ELSE CAST(passageiros_2019.qtde AS INTEGER) END - CASE WHEN CAST(passageiros_1999.qtde AS INTEGER) IS NULL THEN 0 ELSE CAST(passageiros_1999.qtde AS INTEGER) END AS dif_passageiros_2019_1999
,CASE WHEN CAST(passageiros_2023.qtde AS INTEGER) IS NULL THEN 0 ELSE CAST(passageiros_2023.qtde AS INTEGER) END - CASE WHEN CAST(passageiros_2019.qtde AS INTEGER) IS NULL THEN 0 ELSE CAST(passageiros_2019.qtde AS INTEGER) END AS dif_passageiros_2023_2019
,CASE WHEN CAST(passageiros_2023.qtde AS INTEGER) IS NULL THEN 0 ELSE CAST(passageiros_2023.qtde AS INTEGER) END - CASE WHEN CAST(passageiros_1999.qtde AS INTEGER) IS NULL THEN 0 ELSE CAST(passageiros_1999.qtde AS INTEGER) END AS dif_passageiros_2023_1999
,CAST(br_dados_sintese_2010_2022.pop_2010 AS INTEGER) AS pop_2010
,CAST(cidades_pop_2022.pop_2022_total AS INTEGER) AS pop_2022
,CAST(br_dados_sintese_2010_2022.pop_2010_2022_dif AS INTEGER) AS pop_2010_2022_dif
,CAST(br_dados_sintese_2010_2022.pib_2002 AS DECIMAL(18, 2)) AS pib_2002
,CAST(br_dados_sintese_2010_2022.pib_2010 AS DECIMAL(18, 2)) AS pib_2010
,CAST(br_dados_sintese_2010_2022.pib_2010 AS DECIMAL(18, 2)) AS pib_2019
,CAST(br_dados_sintese_2010_2022.pib_2002_2010_percent_evol AS INTEGER) AS pib_2002_2010_percent_evol
,CAST(br_dados_sintese_2010_2022.pib_2010_2019_percent_evol AS INTEGER) AS pib_2010_2019_percent_evol
,CAST(br_dados_sintese_2010_2022.pib_2002_2019_percent_evol AS INTEGER) AS pib_2002_2019_percent_evol
,CONCAT(brasil_regic_2018_hierarquia_geocodigos.mun_hierarquia_18, ' - ', brasil_regic_2018_hierarquia_geocodigos.mun_hierarquia_descr_18) AS hierarquia
FROM cidades

-- Tabela que verifica a quantidade de linhas que comecam ou terminam no municipio
LEFT JOIN (SELECT mun, COUNT(DISTINCT prefixo) AS qtde FROM (SELECT DISTINCT(sig_linha_ori) AS mun, prefixo FROM df WHERE ano = {ano_definicao} AND modalidade = 'RODOVIARIO' AND passageiros_total > 0 UNION ALL SELECT DISTINCT(sig_linha_des) AS mun, prefixo FROM df WHERE ano = {ano_definicao} AND modalidade = 'RODOVIARIO' AND passageiros_total > 0) WHERE mun IS NOT NULL GROUP BY mun ORDER BY mun ASC) AS ligacoes_sao_ori_des
ON ligacoes_sao_ori_des.mun = cidades.mun_nomesigmai

-- Tabela que calcula o total de linhas no municipio
LEFT JOIN (SELECT mun, COUNT(DISTINCT prefixo) AS qtde FROM (SELECT DISTINCT(sig_secao_ori) AS mun, prefixo FROM df WHERE ano = {ano_definicao} AND modalidade = 'RODOVIARIO' AND passageiros_total > 0 UNION ALL SELECT DISTINCT(sig_secao_des) AS mun, prefixo FROM df WHERE ano = {ano_definicao} AND modalidade = 'RODOVIARIO' AND passageiros_total > 0) WHERE mun IS NOT NULL GROUP BY mun ORDER BY mun ASC) AS total_ligacoes
ON total_ligacoes.mun = cidades.mun_nomesigmai

-- Tabela com o total de passageiros movimentados no municipio em linhas RODOVIARIAS no ano de 1999
LEFT JOIN (SELECT mun, SUM(passageiros_total) AS qtde FROM (SELECT sig_secao_ori AS mun, passageiros_total FROM df WHERE ano = 1999 AND passageiros_total > 0 AND modalidade = 'RODOVIARIO' AND sig_secao_ori != sig_secao_des UNION ALL SELECT sig_secao_des AS mun, passageiros_total FROM df WHERE ano = 1999 AND passageiros_total > 0 AND modalidade = 'RODOVIARIO' AND sig_secao_ori != sig_secao_des UNION ALL SELECT sig_secao_ori AS mun, passageiros_total FROM df WHERE ano = 1999 AND passageiros_total > 0 AND modalidade = 'RODOVIARIO' AND sig_secao_ori = sig_secao_des) GROUP BY mun) AS passageiros_1999
ON passageiros_1999.mun = cidades.mun_nomesigmai

-- Tabela com o total de passageiros movimentados no municipio em linhas RODOVIARIAS no ano de 2010
LEFT JOIN (SELECT mun, SUM(passageiros_total) AS qtde FROM (SELECT sig_secao_ori AS mun, passageiros_total FROM df WHERE ano = 2010 AND passageiros_total > 0 AND modalidade = 'RODOVIARIO' AND sig_secao_ori != sig_secao_des UNION ALL SELECT sig_secao_des AS mun, passageiros_total FROM df WHERE ano = 2010 AND passageiros_total > 0 AND modalidade = 'RODOVIARIO' AND sig_secao_ori != sig_secao_des UNION ALL SELECT sig_secao_ori AS mun, passageiros_total FROM df WHERE ano = 2010 AND passageiros_total > 0 AND modalidade = 'RODOVIARIO' AND sig_secao_ori = sig_secao_des) GROUP BY mun) AS passageiros_2010
ON passageiros_2010.mun = cidades.mun_nomesigmai

-- Tabela com o total de passageiros movimentados no municipio em linhas RODOVIARIAS no ano de 2019
LEFT JOIN (SELECT mun, SUM(passageiros_total) AS qtde FROM (SELECT sig_secao_ori AS mun, passageiros_total FROM df WHERE ano = 2019 AND passageiros_total > 0 AND modalidade = 'RODOVIARIO' AND sig_secao_ori != sig_secao_des UNION ALL SELECT sig_secao_des AS mun, passageiros_total FROM df WHERE ano = 2019 AND passageiros_total > 0 AND modalidade = 'RODOVIARIO' AND sig_secao_ori != sig_secao_des UNION ALL SELECT sig_secao_ori AS mun, passageiros_total FROM df WHERE ano = 2019 AND passageiros_total > 0 AND modalidade = 'RODOVIARIO' AND sig_secao_ori = sig_secao_des) GROUP BY mun) AS passageiros_2019
ON passageiros_2019.mun = cidades.mun_nomesigmai

-- Tabela com o total de passageiros movimentados no municipio em linhas RODOVIARIAS no ano de 2023
LEFT JOIN (SELECT mun, SUM(passageiros_total) AS qtde FROM (SELECT sig_secao_ori AS mun, passageiros_total FROM df WHERE ano = {ano_definicao} AND passageiros_total > 0 AND modalidade = 'RODOVIARIO' AND sig_secao_ori != sig_secao_des UNION ALL SELECT sig_secao_des AS mun, passageiros_total FROM df WHERE ano = {ano_definicao} AND passageiros_total > 0 AND modalidade = 'RODOVIARIO' AND sig_secao_ori != sig_secao_des UNION ALL SELECT sig_secao_ori AS mun, passageiros_total FROM df WHERE ano = {ano_definicao} AND passageiros_total > 0 AND modalidade = 'RODOVIARIO' AND sig_secao_ori = sig_secao_des) GROUP BY mun) AS passageiros_2023
ON passageiros_2023.mun = cidades.mun_nomesigmai

-- Tabela com o total de passageiros movimentados no municipio em linhas RODOVIARIAS entre os anos de 1999 e 2023
LEFT JOIN (SELECT mun, SUM(passageiros_total) AS qtde FROM (SELECT sig_secao_ori AS mun, passageiros_total FROM df WHERE passageiros_total > 0 AND modalidade = 'RODOVIARIO' AND sig_secao_ori != sig_secao_des UNION ALL SELECT sig_secao_des AS mun, passageiros_total FROM df WHERE passageiros_total > 0 AND modalidade = 'RODOVIARIO' AND sig_secao_ori != sig_secao_des UNION ALL SELECT sig_secao_des AS mun, passageiros_total FROM df WHERE passageiros_total > 0 AND modalidade = 'RODOVIARIO' AND sig_secao_ori = sig_secao_des) ASC GROUP BY mun ORDER BY mun) AS passageiros_1999_2023
ON passageiros_1999_2023.mun = cidades.mun_nomesigmai

-- Tabela que pega a populacao conforme o Censo Demografico de 2022
LEFT JOIN cidades_pop_2022 AS cidades_pop_2022
ON cidades_pop_2022.geocodigo = cidades.geocodigo

-- Tabela que pega a hierarquia urban aconforme o estudo REGIC 2018 considerando a escala municipal ou seja desfazendo APs e afins
LEFT JOIN brasil_regic_2018_hierarquia_geocodigos AS brasil_regic_2018_hierarquia_geocodigos
ON brasil_regic_2018_hierarquia_geocodigos.geocodigo = cidades.geocodigo

-- Tabela que pega dados referentes a populacao em 2010 conforme o Censo daquele ano e os dados de PIB para varios anoss
LEFT JOIN br_dados_sintese_2010_2022 AS br_dados_sintese_2010_2022
ON br_dados_sintese_2010_2022.geocodigo = cidades.geocodigo

-- Tabela que pega a qtde de linhas do transporte rodoviário interestadual
--                                              LEFT JOIN (SELECT TRIM(municipio_uf) AS mun, geocodigo, localidade_ponto, COUNT(DISTINCT(prefixo)) AS qtde FROM (SELECT a.*, b.geocodigo, CONCAT('POINT(',c.mun_cidade_longitude, ' ', c.mun_cidade_latitude, ')') AS localidade_ponto FROM esquemaop AS a LEFT JOIN diciov3full AS b ON TRIM(CONCAT(b.municipio_nome, '/', b.municipio_uf)) = TRIM(a.municipio_uf) LEFT JOIN cidades AS c ON c.geocodigo = b.geocodigo WHERE processamento = 20240929) WHERE geocodigo IS NOT NULL AND TRIM(municipio_uf) LIKE '%/SC%' GROUP BY TRIM(municipio_uf), geocodigo, localidade_ponto) AS antt ON antt.geocodigo = cidades.geocodigo

-- Filtro na tabela cidades para o estado de SC
WHERE cidades.uf_sigla = 'SC'
""")

# df_trp_sc_tabela_sintese_org_espacial.repartition(1).write.csv(f"./d_painel/{get_time()}_16_trp_sc_tabela_sintese_org_espacial/", header = True, sep = ";", encoding = "utf-8")

# .to_clipboard(index = False, sep = ";", header = True, decimal = ",")

In [6]:
# 2010
df_trp_sc_tabela_sintese_org_espacial.select(["mun_nomesigmai", "qtde_linhas", "qtde_linhas_terminais"]).show(295, False)

+------------------------------+-----------+---------------------+
|mun_nomesigmai                |qtde_linhas|qtde_linhas_terminais|
+------------------------------+-----------+---------------------+
|ABDON BATISTASC               |4          |4                    |
|ABELARDO LUZSC                |12         |1                    |
|AGROLÂNDIASC                  |2          |1                    |
|AGRONÔMICASC                  |11         |null                 |
|ÁGUA DOCESC                   |5          |1                    |
|ÁGUAS DE CHAPECÓSC            |4          |null                 |
|ÁGUAS FRIASSC                 |null       |null                 |
|ÁGUAS MORNASSC                |12         |null                 |
|ALFREDO WAGNERSC              |11         |2                    |
|ALTO BELA VISTASC             |null       |null                 |
|ANCHIETASC                    |4          |null                 |
|ANGELINASC                    |6          |6                 

In [4]:
# 2023
df_trp_sc_tabela_sintese_org_espacial.select(["mun_nomesigmai", "qtde_linhas", "qtde_linhas_terminais"]).show(295, False)

+------------------------------+-----------+---------------------+
|mun_nomesigmai                |qtde_linhas|qtde_linhas_terminais|
+------------------------------+-----------+---------------------+
|ABDON BATISTASC               |null       |null                 |
|ABELARDO LUZSC                |1          |1                    |
|AGROLÂNDIASC                  |1          |null                 |
|AGRONÔMICASC                  |2          |null                 |
|ÁGUA DOCESC                   |1          |1                    |
|ÁGUAS DE CHAPECÓSC            |5          |3                    |
|ÁGUAS FRIASSC                 |1          |null                 |
|ÁGUAS MORNASSC                |1          |null                 |
|ALFREDO WAGNERSC              |4          |null                 |
|ALTO BELA VISTASC             |null       |null                 |
|ANCHIETASC                    |2          |1                    |
|ANGELINASC                    |null       |null              

In [ ]:
df_trp_sc_tabela_sintese_org_espacial.show(10, False)

### 17 Linhas por empresa para SIG - SEM demanda, regioes, total pax etc

In [ ]:
# LINHAS
linhas_por_empresa = spark.sql("""
SELECT DISTINCT(*) FROM
(
SELECT
ano,
df.empresa_codigo,
aliasempresas.empresa_alias,
prefixo,
prefixo_linha,
prefixo_ramal,
modalidade,
sig_linha_ori,
sig_linha_des,
sie_linha_ori_des AS linha_sie,
sig_linha_ori_des AS linha_ibge,
CONCAT('LINESTRING(', ca.mun_cidade_longitude, ' ', ca.mun_cidade_latitude, ', ', cb.mun_cidade_longitude, ' ', cb.mun_cidade_latitude, ')') AS sig_linha_ori_des_wkt
FROM
df
LEFT JOIN aliasempresas ON aliasempresas.empresa_codigo = df.empresa_codigo
LEFT JOIN cidades AS ca ON ca.mun_nomesigmai = df.sig_linha_ori
LEFT JOIN cidades AS cb ON cb.mun_nomesigmai = df.sig_linha_des
WHERE passageiros_total > 0
)
ORDER BY ano ASC, CAST(empresa_codigo AS INTEGER) ASC, CAST(prefixo_linha AS INTEGER) ASC, CAST(prefixo_ramal AS INTEGER) ASC
""")

linhas_por_empresa.repartition(1).write.csv(f"./d_painel/{get_time()}_17_trp_sc_wkt_tabela_linhas_por_empresa_qgis/", sep = ";", encoding = "utf-8", header = True)
# linhas_por_empresa.show(1000, False)

In [ ]:
# LINHAS FULL MESMO SEM PASSAGEIRO!
linhas_por_empresa = spark.sql("""
SELECT DISTINCT(*) FROM
(
SELECT
ano,
df.empresa_codigo,
aliasempresas.empresa_alias,
prefixo,
prefixo_linha,
prefixo_ramal,
modalidade,
sig_linha_ori,
sig_linha_des,
sie_linha_ori_des AS linha_sie,
sig_linha_ori_des AS linha_ibge,
CONCAT('LINESTRING(', ca.mun_cidade_longitude, ' ', ca.mun_cidade_latitude, ', ', cb.mun_cidade_longitude, ' ', cb.mun_cidade_latitude, ')') AS sig_linha_ori_des_wkt
FROM
df
LEFT JOIN aliasempresas ON aliasempresas.empresa_codigo = df.empresa_codigo
LEFT JOIN cidades AS ca ON ca.mun_nomesigmai = df.sig_linha_ori
LEFT JOIN cidades AS cb ON cb.mun_nomesigmai = df.sig_linha_des
)
ORDER BY ano ASC, CAST(empresa_codigo AS INTEGER) ASC, CAST(prefixo_linha AS INTEGER) ASC, CAST(prefixo_ramal AS INTEGER) ASC
""")

linhas_por_empresa.repartition(1).write.csv(f"./d_painel/{get_time()}_17_trp_sc_wkt_tabela_linhas_por_empresa_qgis_full_com_e_sem_pax/", sep = ";", encoding = "utf-8", header = True)
# linhas_por_empresa.show(1000, False)

### renomear 1

In [ ]:
# Renomeando e movendo os arquivos para a pasta d_painel raiz
renomearRaiz()

### 18 - Tabela síntese dos indicadores por linha do TRP catarinense

In [ ]:
# import pandas as pd
from pyspark.sql.functions import *
# santa_catarina_prefixos_1999_2023 = spark.read.csv("./d_painel/trp_sc_prefixos_1999_2023.csv", header = True, sep = ";")
# santa_catarina_prefixos_1999_2023 = santa_catarina_prefixos_1999_2023.withColumn("ano_prefixo", concat(santa_catarina_prefixos_1999_2023["ano"], lit("_"), santa_catarina_prefixos_1999_2023["prefixo"]))
# santa_catarina_prefixos_1999_2023.createOrReplaceTempView("santa_catarina_prefixos_1999_2023")
# santa_catarina_prefixos_1999_2023.show(10, False)

movimentacao_full_parquet_single_file = spark.read.parquet("./d_painel/trp_sc_movimentacao_full_parquet_single_file.parquet")
# movimentacao_full_parquet_single_file = movimentacao_full_parquet_single_file.withColumn("ano_prefixo_empresacod", concat(movimentacao_full_parquet_single_file["ano"], lit("_"), movimentacao_full_parquet_single_file["prefixo"], lit("_"), movimentacao_full_parquet_single_file["empresa_codigo"]))
movimentacao_full_parquet_single_file.createOrReplaceTempView("movimentacao_full_parquet_single_file")
# movimentacao_full_parquet_single_file.show(10, False)

tb_extensao_linhas = spark.sql("SELECT ano_prefixo_empresacod, MAX(CAST(REPLACE(km, ',', '.') AS FLOAT)) AS extenmax FROM movimentacao_full_parquet_single_file WHERE GROUP BY ano_prefixo_empresacod ORDER BY extenmax DESC")
tb_extensao_linhas.createOrReplaceTempView("tb_extensao_linhas")
# tb_extensao_linhas.show(10, False)

tb_viagens_realizadas = spark.read.csv("./d_painel/trp_sc_viagens_realizadas.csv", header = True, sep = ";")
tb_viagens_realizadas = tb_viagens_realizadas.withColumn("ano_prefixo_empresacod", concat(tb_viagens_realizadas["ano"], lit("_"), tb_viagens_realizadas["linha"], lit("_"), tb_viagens_realizadas["empresa_cod"]))
tb_viagens_realizadas.createOrReplaceTempView("tb_viagens_realizadas")
# tb_viagens_realizadas.show(10, False)

# santa_catarina_prefixos_convenio_antt = spark.read.csv("./d_painel/trp_sc_prefixos_convenio_antt.csv", header = True, sep = ";")
# santa_catarina_prefixos_convenio_antt.createOrReplaceTempView("santa_catarina_prefixos_convenio_antt")
# santa_catarina_prefixos_convenio_antt.show(10, False)

tb_qtde_secoes_linhas = spark.sql("SELECT ano_prefixo_empresacod, COUNT(*) AS qtde_secoes FROM movimentacao_full_parquet_single_file GROUP BY ano_prefixo_empresacod")
tb_qtde_secoes_linhas.createOrReplaceTempView("tb_qtde_secoes_linhas")
# tb_qtde_secoes_linhas.show(10, False)

tb_empresa_alias = spark.read.csv("../SIE/_dicionarios/dicionario_empresas_alias.csv", sep = ";", header = True)
tb_empresa_alias.createOrReplaceTempView("tb_empresa_alias")
# tb_empresa_alias.show(10, False)

In [ ]:
tabelasintese = spark.sql("""
SELECT
    SUBSTRING_INDEX(ano_prefixo_empresacod, '_', 1) AS ano
    , e.empresa_alias AS empresa
    , SUBSTRING_INDEX(ano_prefixo_empresacod, '_', -1) AS empresa_cod
    , SUBSTRING_INDEX(SUBSTRING_INDEX(ano_prefixo_empresacod, '_', -2), '_', 1) AS prefixo
    , SUBSTRING_INDEX(SUBSTRING_INDEX(SUBSTRING_INDEX(ano_prefixo_empresacod, '_', -2), '_', 1), '-', 1) AS prefixo_linha
    , SUBSTRING_INDEX(SUBSTRING_INDEX(SUBSTRING_INDEX(ano_prefixo_empresacod, '_', -2), '_', 1), '-', -1) AS prefixo_ramal
    , tabelasintese.*
FROM
(
    SELECT
        ano_prefixo_empresacod
        , prefixo_km
        , CAST(qtde_secoes AS INTEGER) AS qtde_secoes
        , CAST(passageiros AS INTEGER) AS passageiros
        , CAST(passageiros_km AS INTEGER) AS passageiros_km
        , CAST((lugares_ofertados * prefixo_km) AS INTEGER) AS passageiros_km_ideal
        , CAST(lugares_ofertados * (passageiros_km / (lugares_ofertados * prefixo_km)) AS INTEGER) AS pax_equiv
        , CAST(viagens AS INTEGER) AS viagens
        , CAST(lugares_ofertados AS INTEGER) AS lugares_ofertados
        , ROUND((passageiros_km / (lugares_ofertados * prefixo_km)) * 100, 2) AS coef_aprov
        , ROUND((lugares_ofertados * (passageiros_km / (lugares_ofertados * prefixo_km)) / viagens), 2) AS coef_ocup
    FROM (
        SELECT
            a.*
            , b.viagens
            , b.lugares_ofertados
            , c.extenmax AS prefixo_km
            , d.qtde_secoes
        FROM
            (SELECT ano_prefixo_empresacod, SUM(passageiros_total) AS passageiros, SUM(pax_km) AS passageiros_km FROM movimentacao_full_parquet_single_file WHERE passageiros_total > 0 GROUP BY ano_prefixo_empresacod) as a

        LEFT JOIN tb_viagens_realizadas AS b ON b.ano_prefixo_empresacod = a.ano_prefixo_empresacod

        LEFT JOIN tb_extensao_linhas AS c ON c.ano_prefixo_empresacod = a.ano_prefixo_empresacod

        LEFT JOIN tb_qtde_secoes_linhas AS d ON d.ano_prefixo_empresacod = a.ano_prefixo_empresacod

        )
) AS tabelasintese
LEFT JOIN tb_empresa_alias AS e ON e.empresa_codigo = SUBSTRING_INDEX(ano_prefixo_empresacod, '_', -1)
ORDER BY
      CAST(ano AS INTEGER) ASC
    , CAST(empresa_cod AS INTEGER) ASC
    , CAST(prefixo_linha AS INTEGER) ASC
    , CAST(prefixo_ramal AS INTEGER) ASC
""")
# tabelasintese.show(10, False)
tabelasintese.repartition(1).write.csv(f"d_painel/{get_time()}_18_trp_sc_tabela_sintese_indic_linhas/", sep = ";", header = True, encoding = "utf-8")

In [ ]:
# Renomeando e movendo os arquivos para a pasta d_painel raiz
renomearRaiz()

### 19 - Fluxo de passageiros entre seções no formato WKT e desagregado por total, modalidade URBANO e modalidade RODOVIARIO

In [ ]:
for modalidade in ("", "RODOVIARIO", "URBANO"):
    import geopandas as gpd
    from shapely.geometry import LineString, Point
    from shapely import wkt
    df = spark.sql(f"""
    SELECT *, SUBSTRING_INDEX(origem_destino, ' - ', 1) AS origem, SUBSTRING_INDEX(origem_destino, ' - ', -1) AS destino
    FROM (
    SELECT
        ano
        , CASE 
            WHEN LEAST(origem, destino) = origem THEN CONCAT(origem, ' - ', destino)
            ELSE CONCAT(destino, ' - ', origem)
        END AS origem_destino
        , SUM(passageiros_transportados) AS passageiros_transportados
        , FIRST(wkt) AS wkt
    FROM
        (SELECT ano, sig_secao_ori AS origem, sig_secao_des AS destino, sig_secao_ori_des_wkt AS wkt, passageiros_total AS passageiros_transportados FROM data WHERE
        passageiros_total > 0
        AND prefixo NOT IN ('864-0', '865-0', '867-0', '868-0', '869-0', '870-0', '871-0', '872-0', '873-0', '874-0', '875-0', '876-0', '877-0', '878-0', '879-0', '880-0', '881-0', '882-0', '883-0', '884-0', '885-0', '886-0', '887-0', '888-0', '889-0', '890-0', '891-0', '892-0', '893-0', '894-0', '895-0', '896-0', '897-0', '898-0', '899-0', '900-0', '901-0', '902-0', '903-0', '904-0', '905-0', '906-0', '907-0', '908-0', '909-0', '910-0', '911-0', '912-0', '913-0', '914-0', '915-0', '916-0', '917-0', '918-0', '919-0', '920-0', '921-0', '922-0', '923-0', '924-0', '925-0', '926-0', '927-0', '928-0', '929-0', '930-0', '931-0', '932-0', '933-0', '934-0', '935-0', '936-0', '937-0', '938-0', '939-0', '940-0', '941-0', '942-0', '943-0', '944-0', '945-0', '946-0', '947-0', '948-0', '949-0', '950-0', '951-0', '952-0', '953-0', '954-0', '955-0', '956-0', '957-0', '958-0', '959-0', '960-0', '961-0', '962-0', '963-0', '964-0', '965-0', '966-0', '967-0', '968-0', '969-0', '970-0', '971-0', '972-0', '976-0', '977-0', '979-0', '980-0', '981-0')
        AND sig_secao_ori != sig_secao_des
        AND modalidade LIKE '%{modalidade}%'
        )
    GROUP BY
        ano,
        CASE 
            WHEN LEAST(origem, destino) = origem THEN CONCAT(origem, ' - ', destino)
            ELSE CONCAT(destino, ' - ', origem)
        END
    )
    ORDER BY
        ano ASC, passageiros_transportados DESC
    """)
    if modalidade == "":
        # salva CSV com o fluxo total indepentente da modalidade
        df.toPandas().to_csv("./d_painel/trp_sc_wkt_fluxos_secoes.csv", index = False, sep = ";", encoding = "utf-8")
        # salva SHP (shapefile) com o fluxo total indepentente da modalidade
        df_wkt_a = df.toPandas()
        df_wkt_a["wkt"] = df_wkt_a["wkt"].apply(wkt.loads)
        df_wkt_b = gpd.GeoDataFrame(df_wkt_a[["ano", "origem_destino", "origem", "destino", "passageiros_transportados"]], geometry = df_wkt_a["wkt"], crs = 'epsg:4674')
        df_wkt_b.rename(columns = {"ano" : "ano", "origem_destino" : "ori_des", "origem" : "ori", "destino" : "des", "passageiros_transportados" : "pax"}, inplace = True)
        df_wkt_b.to_file(f"./d_painel/trp_sc_wkt_fluxos_secoes.shp", encoding = "utf-8")
    else:
        # salva CSV com o fluxo conforme modalidade
        df.toPandas().to_csv(f"./d_painel/trp_sc_wkt_fluxos_secoes_{modalidade}.csv", index = False, sep = ";", encoding = "utf-8")
        # salva SHP (shapefile) com o fluxo conforme modalidade
        df_wkt_a = df.toPandas()
        df_wkt_a["wkt"] = df_wkt_a["wkt"].apply(wkt.loads)
        df_wkt_b = gpd.GeoDataFrame(df_wkt_a[["ano", "origem_destino", "origem", "destino", "passageiros_transportados"]], geometry = df_wkt_a["wkt"], crs = 'epsg:4674')
        df_wkt_b.rename(columns = {"ano" : "ano", "origem_destino" : "ori_des", "origem" : "ori", "destino" : "des", "passageiros_transportados" : "pax"}, inplace = True)
        df_wkt_b.to_file(f"./d_painel/trp_sc_wkt_fluxos_secoes_{modalidade}.shp", encoding = "utf-8")

### 20 - Viagens por seção

In [ ]:
trp_sc_tabela_sintese_indic_linhas = spark.read.csv("./d_painel/trp_sc_tabela_sintese_indic_linhas.csv", sep = ";", header = True)
trp_sc_tabela_sintese_indic_linhas.createOrReplaceTempView("trp_sc_tabela_sintese_indic_linhas")

In [ ]:
for ano in (2000, 2005, 2011, 2015, 2019, 2021, 2023):
    tb_viagens_por_secao = spark.sql(f"""
    SELECT
    CASE WHEN LEAST(sig_secao_ori, sig_secao_des) = sig_secao_ori THEN CONCAT(sig_secao_ori, ' - ', sig_secao_des) ELSE CONCAT(sig_secao_des, ' - ', sig_secao_ori) END AS origem_destino
    , SUBSTRING_INDEX(CASE WHEN LEAST(sig_secao_ori, sig_secao_des) = sig_secao_ori THEN CONCAT(sig_secao_ori, ' - ', sig_secao_des) ELSE CONCAT(sig_secao_des, ' - ', sig_secao_ori) END, ' - ', 1) AS origem
    , SUBSTRING_INDEX(CASE WHEN LEAST(sig_secao_ori, sig_secao_des) = sig_secao_ori THEN CONCAT(sig_secao_ori, ' - ', sig_secao_des) ELSE CONCAT(sig_secao_des, ' - ', sig_secao_ori) END, ' - ', -1) AS destino
    , FIRST(sig_secao_ori_des_wkt) AS wkt
    , SUM(viagens_total_ida_e_volta) AS viagens_total_ida_e_volta
    , SUM(lugares_ofertados_total_ida_e_volta) AS lugares_ofertados_total_ida_e_volta
    , SUM(viagens_media_semanal_ida_e_volta) AS viagens_media_semanal_ida_e_volta
    , SUM(lugares_ofertados_media_semanal_ida_e_volta) AS lugares_ofertados_media_semanal_ida_e_volta
    , SUM(viagens_media_diaria_ida_e_volta) AS viagens_media_diaria_ida_e_volta
    , CASE
        WHEN CAST(SUM(viagens_total_ida_e_volta) / 365 AS INTEGER) <= 2 THEN 'Até 1 viagem diária por sentido'
        WHEN CAST(SUM(viagens_total_ida_e_volta) / 365 AS INTEGER) > 2 AND CAST(SUM(viagens_total_ida_e_volta) / 365 AS INTEGER) <= 6 THEN 'De 2 a 3 viagens diárias por sentido'
        WHEN CAST(SUM(viagens_total_ida_e_volta) / 365 AS INTEGER) > 6 AND CAST(SUM(viagens_total_ida_e_volta) / 365 AS INTEGER) <= 10 THEN 'De 4 a 5 viagens diárias por sentido'
        WHEN CAST(SUM(viagens_total_ida_e_volta) / 365 AS INTEGER) > 10 AND CAST(SUM(viagens_total_ida_e_volta) / 365 AS INTEGER) <= 20 THEN 'De 6 a 10 viagens diárias por sentido'
        WHEN CAST(SUM(viagens_total_ida_e_volta) / 365 AS INTEGER) > 20 THEN 'Mais que 10 viagens diárias por sentido'
    END AS viagens_media_diaria_ida_e_volta_categoria
    FROM
        (SELECT
        sig_secao_ori_des
        , sig_secao_ori
        , sig_secao_des
        , sig_secao_ori_des_wkt
        , CAST(SUM(viagens) AS INTEGER) AS viagens_total_ida_e_volta
        , CAST(SUM(lugares_ofertados) AS INTEGER) AS lugares_ofertados_total_ida_e_volta
        , CAST(SUM(viagens) / 52 AS INTEGER) AS viagens_media_semanal_ida_e_volta
        , CAST(SUM(lugares_ofertados) / 52 AS INTEGER) AS lugares_ofertados_media_semanal_ida_e_volta
        , CAST(SUM(viagens) / 365 AS INTEGER) AS viagens_media_diaria_ida_e_volta
        , CAST(SUM(lugares_ofertados) / 365 AS INTEGER) AS lugares_ofertados_media_diaria_ida_e_volta
        , CASE
            WHEN CAST(SUM(viagens) / 365 AS INTEGER) <= 2 THEN 'Até 1 viagem diária por sentido'
            WHEN CAST(SUM(viagens) / 365 AS INTEGER) > 2 AND CAST(SUM(viagens) / 365 AS INTEGER) <= 10 THEN 'De 2 a 5 viagens diárias por sentido'
            WHEN CAST(SUM(viagens) / 365 AS INTEGER) > 10 AND CAST(SUM(viagens) / 365 AS INTEGER) <= 20 THEN 'De 6 a 10 viagens diárias por sentido'
            WHEN CAST(SUM(viagens) / 365 AS INTEGER) > 20 THEN 'Mais que 10 viagens diárias por sentido'
        END AS viagens_media_diaria_ida_e_volta_categoria
        FROM (
            SELECT DISTINCT(*) FROM (
                SELECT
                    data.ano_prefixo_empresacod
                    , data.sig_secao_ori_des
                    , data.sig_secao_ori
                    , data.sig_secao_des
                    , data.sig_secao_ori_des_wkt
                    , indicadores.viagens
                    , indicadores.lugares_ofertados
                FROM
                    data
                LEFT JOIN
                    trp_sc_tabela_sintese_indic_linhas AS indicadores ON indicadores.ano_prefixo_empresacod = data.ano_prefixo_empresacod
                WHERE
                    data.ano = {ano}
                    AND passageiros_total > 0
                    AND data.sig_secao_ori != data.sig_secao_des
                    --AND modalidade = 'RODOVIARIO'
            )
        )
        GROUP BY sig_secao_ori_des, sig_secao_ori_des_wkt, sig_secao_ori, sig_secao_des)
    GROUP BY CASE WHEN LEAST(sig_secao_ori, sig_secao_des) = sig_secao_ori THEN CONCAT(sig_secao_ori, ' - ', sig_secao_des) ELSE CONCAT(sig_secao_des, ' - ', sig_secao_ori) END
    ORDER BY viagens_total_ida_e_volta DESC
    """)
    # tb_viagens_por_secao.show(10, False)
    tb_viagens_por_secao.toPandas().to_csv(f"./d_painel/tb_viagens_por_secao_{ano}.csv", index = False, header = True, sep = ";", encoding = "utf-8")

### 21 - Viagens por município

In [ ]:
# carrega tabela que tem indicadores de viagens por linha
trp_sc_tabela_sintese_indic_linhas = spark.read.csv("./d_painel/trp_sc_tabela_sintese_indic_linhas.csv", sep = ";", header = True)
trp_sc_tabela_sintese_indic_linhas.createOrReplaceTempView("trp_sc_tabela_sintese_indic_linhas")

In [ ]:
tabela_linhas_por_municipio = spark.sql("""
-- Base para obtenção das linhas por ano, modalidade e município
SELECT CONCAT(ano, '_', prefixo, '_', empresa_codigo) AS ano_prefixo_empresacod, ano, modalidade, sig_secao_ori AS municipio FROM data WHERE
passageiros_total > 0 AND prefixo NOT IN ('864-0', '865-0', '867-0', '868-0', '869-0', '870-0', '871-0', '872-0', '873-0', '874-0', '875-0', '876-0', '877-0', '878-0', '879-0', '880-0', '881-0', '882-0', '883-0', '884-0', '885-0', '886-0', '887-0', '888-0', '889-0', '890-0', '891-0', '892-0', '893-0', '894-0', '895-0', '896-0', '897-0', '898-0', '899-0', '900-0', '901-0', '902-0', '903-0', '904-0', '905-0', '906-0', '907-0', '908-0', '909-0', '910-0', '911-0', '912-0', '913-0', '914-0', '915-0', '916-0', '917-0', '918-0', '919-0', '920-0', '921-0', '922-0', '923-0', '924-0', '925-0', '926-0', '927-0', '928-0', '929-0', '930-0', '931-0', '932-0', '933-0', '934-0', '935-0', '936-0', '937-0', '938-0', '939-0', '940-0', '941-0', '942-0', '943-0', '944-0', '945-0', '946-0', '947-0', '948-0', '949-0', '950-0', '951-0', '952-0', '953-0', '954-0', '955-0', '956-0', '957-0', '958-0', '959-0', '960-0', '961-0', '962-0', '963-0', '964-0', '965-0', '966-0', '967-0', '968-0', '969-0', '970-0', '971-0', '972-0', '976-0', '977-0', '979-0', '980-0', '981-0')
UNION ALL
SELECT CONCAT(ano, '_', prefixo, '_', empresa_codigo) AS ano_prefixo_empresacod, ano, modalidade, sig_secao_des AS municipio FROM data WHERE
passageiros_total > 0 AND prefixo NOT IN ('864-0', '865-0', '867-0', '868-0', '869-0', '870-0', '871-0', '872-0', '873-0', '874-0', '875-0', '876-0', '877-0', '878-0', '879-0', '880-0', '881-0', '882-0', '883-0', '884-0', '885-0', '886-0', '887-0', '888-0', '889-0', '890-0', '891-0', '892-0', '893-0', '894-0', '895-0', '896-0', '897-0', '898-0', '899-0', '900-0', '901-0', '902-0', '903-0', '904-0', '905-0', '906-0', '907-0', '908-0', '909-0', '910-0', '911-0', '912-0', '913-0', '914-0', '915-0', '916-0', '917-0', '918-0', '919-0', '920-0', '921-0', '922-0', '923-0', '924-0', '925-0', '926-0', '927-0', '928-0', '929-0', '930-0', '931-0', '932-0', '933-0', '934-0', '935-0', '936-0', '937-0', '938-0', '939-0', '940-0', '941-0', '942-0', '943-0', '944-0', '945-0', '946-0', '947-0', '948-0', '949-0', '950-0', '951-0', '952-0', '953-0', '954-0', '955-0', '956-0', '957-0', '958-0', '959-0', '960-0', '961-0', '962-0', '963-0', '964-0', '965-0', '966-0', '967-0', '968-0', '969-0', '970-0', '971-0', '972-0', '976-0', '977-0', '979-0', '980-0', '981-0')
ORDER BY ano_prefixo_empresacod
""")
tabela_linhas_por_municipio.createOrReplaceTempView("tabela_linhas_por_municipio")

In [ ]:
tabela_linhas_por_municipio_ref = spark.sql("""
-- Tabela que contém as linhas em operação por município segundo ano e modalidade escolhidos, removendo linhas duplicadas em função das seções
SELECT DISTINCT(*) FROM tabela_linhas_por_municipio
ORDER BY ano_prefixo_empresacod
""")
tabela_linhas_por_municipio_ref.createOrReplaceTempView("tabela_linhas_por_municipio_ref")

In [ ]:
indicadores = spark.sql("""
-- Obtém o núnmero de viagens por ano, modalidade, linha e município a partir da união das tabelas 'trp_sc_tabela_sintese_indic_linhas' e 'tabela_linhas_por_municipio_ref'
SELECT a.ano_prefixo_empresacod, a.ano, a.modalidade, a.municipio, b.viagens FROM tabela_linhas_por_municipio_ref AS a
LEFT JOIN (SELECT ano_prefixo_empresacod, ano, viagens, lugares_ofertados FROM trp_sc_tabela_sintese_indic_linhas) AS b
ON a.ano_prefixo_empresacod = b.ano_prefixo_empresacod
""")
indicadores.createOrReplaceTempView("indicadores")

In [ ]:
tb_viagens_por_cidade_total_rodoviario_urbano = spark.sql("""
SELECT ano, modalidade, municipio, CAST(SUM(total_viagens_ano) AS INTEGER) AS total_viagens_ano, CAST(SUM(media_viagens_diarias) AS INTEGER) AS media_viagens_diarias FROM (
-- Obtém a soma das viagens totais independentemente da linha por ano, modalidade e município (agrega municipio)
SELECT
        ano
        , modalidade
        , municipio
        , CAST(SUM(viagens) AS INTEGER) AS total_viagens_ano
        , CAST(SUM(viagens) / 365 AS INTEGER) AS media_viagens_diarias
FROM indicadores

-- Agrupa dados por município para somar
GROUP BY ano, modalidade, municipio

UNION ALL

-- Obtém a soma das viagens totais independentemente da linha por ano, modalidade e município (agrega municipio)
SELECT
        ano
        , 'TOTAL' AS modalidade
        , municipio
        , CAST(SUM(viagens) AS INTEGER) AS total_viagens_ano
        , CAST(SUM(viagens) / 365 AS INTEGER) AS media_viagens_diarias
FROM indicadores

-- Agrupa dados por município para somar
GROUP BY ano, modalidade, municipio)

GROUP BY ano, modalidade, municipio

-- Ordena tabela por município
ORDER BY ano, modalidade, municipio
""")
tb_viagens_por_cidade_total_rodoviario_urbano.createOrReplaceTempView("tb_viagens_por_cidade_total_rodoviario_urbano")

In [ ]:
tb_viagens_por_cidade_total_rodoviario_urbano.repartition(1).write.csv(f"./d_painel/{get_time()}_100_trp_sc_tb_viagens_por_cidade_total_rodoviario_urbano_not_pivot/", header = True, sep = ";", encoding = "utf-8")

In [ ]:
tb_viagens_por_cidade_total_rodoviario_urbano_pivot_tudo = spark.sql("""
SELECT
municipio,
SUM(CASE WHEN ano = 1999 AND modalidade = 'RODOVIARIO' THEN total_viagens_ano ELSE 0 END) AS tot_v_r_1999, SUM(CASE WHEN ano = 1999 AND modalidade = 'URBANO' THEN total_viagens_ano ELSE 0 END) AS tot_v_u_1999, SUM(CASE WHEN ano = 1999 AND modalidade = 'TOTAL' THEN total_viagens_ano ELSE 0 END) AS tot_v_t_1999, SUM(CASE WHEN ano = 1999 AND modalidade = 'RODOVIARIO' THEN media_viagens_diarias ELSE 0 END) AS med_v_d_r_1999, SUM(CASE WHEN ano = 1999 AND modalidade = 'URBANO' THEN media_viagens_diarias ELSE 0 END) AS med_v_d_u_1999, SUM(CASE WHEN ano = 1999 AND modalidade = 'TOTAL' THEN media_viagens_diarias ELSE 0 END) AS med_v_d_t_1999,
SUM(CASE WHEN ano = 2000 AND modalidade = 'RODOVIARIO' THEN total_viagens_ano ELSE 0 END) AS tot_v_r_2000, SUM(CASE WHEN ano = 2000 AND modalidade = 'URBANO' THEN total_viagens_ano ELSE 0 END) AS tot_v_u_2000, SUM(CASE WHEN ano = 2000 AND modalidade = 'TOTAL' THEN total_viagens_ano ELSE 0 END) AS tot_v_t_2000, SUM(CASE WHEN ano = 2000 AND modalidade = 'RODOVIARIO' THEN media_viagens_diarias ELSE 0 END) AS med_v_d_r_2000, SUM(CASE WHEN ano = 2000 AND modalidade = 'URBANO' THEN media_viagens_diarias ELSE 0 END) AS med_v_d_u_2000, SUM(CASE WHEN ano = 2000 AND modalidade = 'TOTAL' THEN media_viagens_diarias ELSE 0 END) AS med_v_d_t_2000,
SUM(CASE WHEN ano = 2001 AND modalidade = 'RODOVIARIO' THEN total_viagens_ano ELSE 0 END) AS tot_v_r_2001, SUM(CASE WHEN ano = 2001 AND modalidade = 'URBANO' THEN total_viagens_ano ELSE 0 END) AS tot_v_u_2001, SUM(CASE WHEN ano = 2001 AND modalidade = 'TOTAL' THEN total_viagens_ano ELSE 0 END) AS tot_v_t_2001, SUM(CASE WHEN ano = 2001 AND modalidade = 'RODOVIARIO' THEN media_viagens_diarias ELSE 0 END) AS med_v_d_r_2001, SUM(CASE WHEN ano = 2001 AND modalidade = 'URBANO' THEN media_viagens_diarias ELSE 0 END) AS med_v_d_u_2001, SUM(CASE WHEN ano = 2001 AND modalidade = 'TOTAL' THEN media_viagens_diarias ELSE 0 END) AS med_v_d_t_2001,
SUM(CASE WHEN ano = 2002 AND modalidade = 'RODOVIARIO' THEN total_viagens_ano ELSE 0 END) AS tot_v_r_2002, SUM(CASE WHEN ano = 2002 AND modalidade = 'URBANO' THEN total_viagens_ano ELSE 0 END) AS tot_v_u_2002, SUM(CASE WHEN ano = 2002 AND modalidade = 'TOTAL' THEN total_viagens_ano ELSE 0 END) AS tot_v_t_2002, SUM(CASE WHEN ano = 2002 AND modalidade = 'RODOVIARIO' THEN media_viagens_diarias ELSE 0 END) AS med_v_d_r_2002, SUM(CASE WHEN ano = 2002 AND modalidade = 'URBANO' THEN media_viagens_diarias ELSE 0 END) AS med_v_d_u_2002, SUM(CASE WHEN ano = 2002 AND modalidade = 'TOTAL' THEN media_viagens_diarias ELSE 0 END) AS med_v_d_t_2002,
SUM(CASE WHEN ano = 2003 AND modalidade = 'RODOVIARIO' THEN total_viagens_ano ELSE 0 END) AS tot_v_r_2003, SUM(CASE WHEN ano = 2003 AND modalidade = 'URBANO' THEN total_viagens_ano ELSE 0 END) AS tot_v_u_2003, SUM(CASE WHEN ano = 2003 AND modalidade = 'TOTAL' THEN total_viagens_ano ELSE 0 END) AS tot_v_t_2003, SUM(CASE WHEN ano = 2003 AND modalidade = 'RODOVIARIO' THEN media_viagens_diarias ELSE 0 END) AS med_v_d_r_2003, SUM(CASE WHEN ano = 2003 AND modalidade = 'URBANO' THEN media_viagens_diarias ELSE 0 END) AS med_v_d_u_2003, SUM(CASE WHEN ano = 2003 AND modalidade = 'TOTAL' THEN media_viagens_diarias ELSE 0 END) AS med_v_d_t_2003,
SUM(CASE WHEN ano = 2004 AND modalidade = 'RODOVIARIO' THEN total_viagens_ano ELSE 0 END) AS tot_v_r_2004, SUM(CASE WHEN ano = 2004 AND modalidade = 'URBANO' THEN total_viagens_ano ELSE 0 END) AS tot_v_u_2004, SUM(CASE WHEN ano = 2004 AND modalidade = 'TOTAL' THEN total_viagens_ano ELSE 0 END) AS tot_v_t_2004, SUM(CASE WHEN ano = 2004 AND modalidade = 'RODOVIARIO' THEN media_viagens_diarias ELSE 0 END) AS med_v_d_r_2004, SUM(CASE WHEN ano = 2004 AND modalidade = 'URBANO' THEN media_viagens_diarias ELSE 0 END) AS med_v_d_u_2004, SUM(CASE WHEN ano = 2004 AND modalidade = 'TOTAL' THEN media_viagens_diarias ELSE 0 END) AS med_v_d_t_2004,
SUM(CASE WHEN ano = 2005 AND modalidade = 'RODOVIARIO' THEN total_viagens_ano ELSE 0 END) AS tot_v_r_2005, SUM(CASE WHEN ano = 2005 AND modalidade = 'URBANO' THEN total_viagens_ano ELSE 0 END) AS tot_v_u_2005, SUM(CASE WHEN ano = 2005 AND modalidade = 'TOTAL' THEN total_viagens_ano ELSE 0 END) AS tot_v_t_2005, SUM(CASE WHEN ano = 2005 AND modalidade = 'RODOVIARIO' THEN media_viagens_diarias ELSE 0 END) AS med_v_d_r_2005, SUM(CASE WHEN ano = 2005 AND modalidade = 'URBANO' THEN media_viagens_diarias ELSE 0 END) AS med_v_d_u_2005, SUM(CASE WHEN ano = 2005 AND modalidade = 'TOTAL' THEN media_viagens_diarias ELSE 0 END) AS med_v_d_t_2005,
SUM(CASE WHEN ano = 2006 AND modalidade = 'RODOVIARIO' THEN total_viagens_ano ELSE 0 END) AS tot_v_r_2006, SUM(CASE WHEN ano = 2006 AND modalidade = 'URBANO' THEN total_viagens_ano ELSE 0 END) AS tot_v_u_2006, SUM(CASE WHEN ano = 2006 AND modalidade = 'TOTAL' THEN total_viagens_ano ELSE 0 END) AS tot_v_t_2006, SUM(CASE WHEN ano = 2006 AND modalidade = 'RODOVIARIO' THEN media_viagens_diarias ELSE 0 END) AS med_v_d_r_2006, SUM(CASE WHEN ano = 2006 AND modalidade = 'URBANO' THEN media_viagens_diarias ELSE 0 END) AS med_v_d_u_2006, SUM(CASE WHEN ano = 2006 AND modalidade = 'TOTAL' THEN media_viagens_diarias ELSE 0 END) AS med_v_d_t_2006,
SUM(CASE WHEN ano = 2007 AND modalidade = 'RODOVIARIO' THEN total_viagens_ano ELSE 0 END) AS tot_v_r_2007, SUM(CASE WHEN ano = 2007 AND modalidade = 'URBANO' THEN total_viagens_ano ELSE 0 END) AS tot_v_u_2007, SUM(CASE WHEN ano = 2007 AND modalidade = 'TOTAL' THEN total_viagens_ano ELSE 0 END) AS tot_v_t_2007, SUM(CASE WHEN ano = 2007 AND modalidade = 'RODOVIARIO' THEN media_viagens_diarias ELSE 0 END) AS med_v_d_r_2007, SUM(CASE WHEN ano = 2007 AND modalidade = 'URBANO' THEN media_viagens_diarias ELSE 0 END) AS med_v_d_u_2007, SUM(CASE WHEN ano = 2007 AND modalidade = 'TOTAL' THEN media_viagens_diarias ELSE 0 END) AS med_v_d_t_2007,
SUM(CASE WHEN ano = 2008 AND modalidade = 'RODOVIARIO' THEN total_viagens_ano ELSE 0 END) AS tot_v_r_2008, SUM(CASE WHEN ano = 2008 AND modalidade = 'URBANO' THEN total_viagens_ano ELSE 0 END) AS tot_v_u_2008, SUM(CASE WHEN ano = 2008 AND modalidade = 'TOTAL' THEN total_viagens_ano ELSE 0 END) AS tot_v_t_2008, SUM(CASE WHEN ano = 2008 AND modalidade = 'RODOVIARIO' THEN media_viagens_diarias ELSE 0 END) AS med_v_d_r_2008, SUM(CASE WHEN ano = 2008 AND modalidade = 'URBANO' THEN media_viagens_diarias ELSE 0 END) AS med_v_d_u_2008, SUM(CASE WHEN ano = 2008 AND modalidade = 'TOTAL' THEN media_viagens_diarias ELSE 0 END) AS med_v_d_t_2008,
SUM(CASE WHEN ano = 2009 AND modalidade = 'RODOVIARIO' THEN total_viagens_ano ELSE 0 END) AS tot_v_r_2009, SUM(CASE WHEN ano = 2009 AND modalidade = 'URBANO' THEN total_viagens_ano ELSE 0 END) AS tot_v_u_2009, SUM(CASE WHEN ano = 2009 AND modalidade = 'TOTAL' THEN total_viagens_ano ELSE 0 END) AS tot_v_t_2009, SUM(CASE WHEN ano = 2009 AND modalidade = 'RODOVIARIO' THEN media_viagens_diarias ELSE 0 END) AS med_v_d_r_2009, SUM(CASE WHEN ano = 2009 AND modalidade = 'URBANO' THEN media_viagens_diarias ELSE 0 END) AS med_v_d_u_2009, SUM(CASE WHEN ano = 2009 AND modalidade = 'TOTAL' THEN media_viagens_diarias ELSE 0 END) AS med_v_d_t_2009,
SUM(CASE WHEN ano = 2010 AND modalidade = 'RODOVIARIO' THEN total_viagens_ano ELSE 0 END) AS tot_v_r_2010, SUM(CASE WHEN ano = 2010 AND modalidade = 'URBANO' THEN total_viagens_ano ELSE 0 END) AS tot_v_u_2010, SUM(CASE WHEN ano = 2010 AND modalidade = 'TOTAL' THEN total_viagens_ano ELSE 0 END) AS tot_v_t_2010, SUM(CASE WHEN ano = 2010 AND modalidade = 'RODOVIARIO' THEN media_viagens_diarias ELSE 0 END) AS med_v_d_r_2010, SUM(CASE WHEN ano = 2010 AND modalidade = 'URBANO' THEN media_viagens_diarias ELSE 0 END) AS med_v_d_u_2010, SUM(CASE WHEN ano = 2010 AND modalidade = 'TOTAL' THEN media_viagens_diarias ELSE 0 END) AS med_v_d_t_2010,
SUM(CASE WHEN ano = 2011 AND modalidade = 'RODOVIARIO' THEN total_viagens_ano ELSE 0 END) AS tot_v_r_2011, SUM(CASE WHEN ano = 2011 AND modalidade = 'URBANO' THEN total_viagens_ano ELSE 0 END) AS tot_v_u_2011, SUM(CASE WHEN ano = 2011 AND modalidade = 'TOTAL' THEN total_viagens_ano ELSE 0 END) AS tot_v_t_2011, SUM(CASE WHEN ano = 2011 AND modalidade = 'RODOVIARIO' THEN media_viagens_diarias ELSE 0 END) AS med_v_d_r_2011, SUM(CASE WHEN ano = 2011 AND modalidade = 'URBANO' THEN media_viagens_diarias ELSE 0 END) AS med_v_d_u_2011, SUM(CASE WHEN ano = 2011 AND modalidade = 'TOTAL' THEN media_viagens_diarias ELSE 0 END) AS med_v_d_t_2011,
SUM(CASE WHEN ano = 2012 AND modalidade = 'RODOVIARIO' THEN total_viagens_ano ELSE 0 END) AS tot_v_r_2012, SUM(CASE WHEN ano = 2012 AND modalidade = 'URBANO' THEN total_viagens_ano ELSE 0 END) AS tot_v_u_2012, SUM(CASE WHEN ano = 2012 AND modalidade = 'TOTAL' THEN total_viagens_ano ELSE 0 END) AS tot_v_t_2012, SUM(CASE WHEN ano = 2012 AND modalidade = 'RODOVIARIO' THEN media_viagens_diarias ELSE 0 END) AS med_v_d_r_2012, SUM(CASE WHEN ano = 2012 AND modalidade = 'URBANO' THEN media_viagens_diarias ELSE 0 END) AS med_v_d_u_2012, SUM(CASE WHEN ano = 2012 AND modalidade = 'TOTAL' THEN media_viagens_diarias ELSE 0 END) AS med_v_d_t_2012,
SUM(CASE WHEN ano = 2013 AND modalidade = 'RODOVIARIO' THEN total_viagens_ano ELSE 0 END) AS tot_v_r_2013, SUM(CASE WHEN ano = 2013 AND modalidade = 'URBANO' THEN total_viagens_ano ELSE 0 END) AS tot_v_u_2013, SUM(CASE WHEN ano = 2013 AND modalidade = 'TOTAL' THEN total_viagens_ano ELSE 0 END) AS tot_v_t_2013, SUM(CASE WHEN ano = 2013 AND modalidade = 'RODOVIARIO' THEN media_viagens_diarias ELSE 0 END) AS med_v_d_r_2013, SUM(CASE WHEN ano = 2013 AND modalidade = 'URBANO' THEN media_viagens_diarias ELSE 0 END) AS med_v_d_u_2013, SUM(CASE WHEN ano = 2013 AND modalidade = 'TOTAL' THEN media_viagens_diarias ELSE 0 END) AS med_v_d_t_2013,
SUM(CASE WHEN ano = 2014 AND modalidade = 'RODOVIARIO' THEN total_viagens_ano ELSE 0 END) AS tot_v_r_2014, SUM(CASE WHEN ano = 2014 AND modalidade = 'URBANO' THEN total_viagens_ano ELSE 0 END) AS tot_v_u_2014, SUM(CASE WHEN ano = 2014 AND modalidade = 'TOTAL' THEN total_viagens_ano ELSE 0 END) AS tot_v_t_2014, SUM(CASE WHEN ano = 2014 AND modalidade = 'RODOVIARIO' THEN media_viagens_diarias ELSE 0 END) AS med_v_d_r_2014, SUM(CASE WHEN ano = 2014 AND modalidade = 'URBANO' THEN media_viagens_diarias ELSE 0 END) AS med_v_d_u_2014, SUM(CASE WHEN ano = 2014 AND modalidade = 'TOTAL' THEN media_viagens_diarias ELSE 0 END) AS med_v_d_t_2014,
SUM(CASE WHEN ano = 2015 AND modalidade = 'RODOVIARIO' THEN total_viagens_ano ELSE 0 END) AS tot_v_r_2015, SUM(CASE WHEN ano = 2015 AND modalidade = 'URBANO' THEN total_viagens_ano ELSE 0 END) AS tot_v_u_2015, SUM(CASE WHEN ano = 2015 AND modalidade = 'TOTAL' THEN total_viagens_ano ELSE 0 END) AS tot_v_t_2015, SUM(CASE WHEN ano = 2015 AND modalidade = 'RODOVIARIO' THEN media_viagens_diarias ELSE 0 END) AS med_v_d_r_2015, SUM(CASE WHEN ano = 2015 AND modalidade = 'URBANO' THEN media_viagens_diarias ELSE 0 END) AS med_v_d_u_2015, SUM(CASE WHEN ano = 2015 AND modalidade = 'TOTAL' THEN media_viagens_diarias ELSE 0 END) AS med_v_d_t_2015,
SUM(CASE WHEN ano = 2016 AND modalidade = 'RODOVIARIO' THEN total_viagens_ano ELSE 0 END) AS tot_v_r_2016, SUM(CASE WHEN ano = 2016 AND modalidade = 'URBANO' THEN total_viagens_ano ELSE 0 END) AS tot_v_u_2016, SUM(CASE WHEN ano = 2016 AND modalidade = 'TOTAL' THEN total_viagens_ano ELSE 0 END) AS tot_v_t_2016, SUM(CASE WHEN ano = 2016 AND modalidade = 'RODOVIARIO' THEN media_viagens_diarias ELSE 0 END) AS med_v_d_r_2016, SUM(CASE WHEN ano = 2016 AND modalidade = 'URBANO' THEN media_viagens_diarias ELSE 0 END) AS med_v_d_u_2016, SUM(CASE WHEN ano = 2016 AND modalidade = 'TOTAL' THEN media_viagens_diarias ELSE 0 END) AS med_v_d_t_2016,
SUM(CASE WHEN ano = 2017 AND modalidade = 'RODOVIARIO' THEN total_viagens_ano ELSE 0 END) AS tot_v_r_2017, SUM(CASE WHEN ano = 2017 AND modalidade = 'URBANO' THEN total_viagens_ano ELSE 0 END) AS tot_v_u_2017, SUM(CASE WHEN ano = 2017 AND modalidade = 'TOTAL' THEN total_viagens_ano ELSE 0 END) AS tot_v_t_2017, SUM(CASE WHEN ano = 2017 AND modalidade = 'RODOVIARIO' THEN media_viagens_diarias ELSE 0 END) AS med_v_d_r_2017, SUM(CASE WHEN ano = 2017 AND modalidade = 'URBANO' THEN media_viagens_diarias ELSE 0 END) AS med_v_d_u_2017, SUM(CASE WHEN ano = 2017 AND modalidade = 'TOTAL' THEN media_viagens_diarias ELSE 0 END) AS med_v_d_t_2017,
SUM(CASE WHEN ano = 2018 AND modalidade = 'RODOVIARIO' THEN total_viagens_ano ELSE 0 END) AS tot_v_r_2018, SUM(CASE WHEN ano = 2018 AND modalidade = 'URBANO' THEN total_viagens_ano ELSE 0 END) AS tot_v_u_2018, SUM(CASE WHEN ano = 2018 AND modalidade = 'TOTAL' THEN total_viagens_ano ELSE 0 END) AS tot_v_t_2018, SUM(CASE WHEN ano = 2018 AND modalidade = 'RODOVIARIO' THEN media_viagens_diarias ELSE 0 END) AS med_v_d_r_2018, SUM(CASE WHEN ano = 2018 AND modalidade = 'URBANO' THEN media_viagens_diarias ELSE 0 END) AS med_v_d_u_2018, SUM(CASE WHEN ano = 2018 AND modalidade = 'TOTAL' THEN media_viagens_diarias ELSE 0 END) AS med_v_d_t_2018,
SUM(CASE WHEN ano = 2019 AND modalidade = 'RODOVIARIO' THEN total_viagens_ano ELSE 0 END) AS tot_v_r_2019, SUM(CASE WHEN ano = 2019 AND modalidade = 'URBANO' THEN total_viagens_ano ELSE 0 END) AS tot_v_u_2019, SUM(CASE WHEN ano = 2019 AND modalidade = 'TOTAL' THEN total_viagens_ano ELSE 0 END) AS tot_v_t_2019, SUM(CASE WHEN ano = 2019 AND modalidade = 'RODOVIARIO' THEN media_viagens_diarias ELSE 0 END) AS med_v_d_r_2019, SUM(CASE WHEN ano = 2019 AND modalidade = 'URBANO' THEN media_viagens_diarias ELSE 0 END) AS med_v_d_u_2019, SUM(CASE WHEN ano = 2019 AND modalidade = 'TOTAL' THEN media_viagens_diarias ELSE 0 END) AS med_v_d_t_2019,
SUM(CASE WHEN ano = 2020 AND modalidade = 'RODOVIARIO' THEN total_viagens_ano ELSE 0 END) AS tot_v_r_2020, SUM(CASE WHEN ano = 2020 AND modalidade = 'URBANO' THEN total_viagens_ano ELSE 0 END) AS tot_v_u_2020, SUM(CASE WHEN ano = 2020 AND modalidade = 'TOTAL' THEN total_viagens_ano ELSE 0 END) AS tot_v_t_2020, SUM(CASE WHEN ano = 2020 AND modalidade = 'RODOVIARIO' THEN media_viagens_diarias ELSE 0 END) AS med_v_d_r_2020, SUM(CASE WHEN ano = 2020 AND modalidade = 'URBANO' THEN media_viagens_diarias ELSE 0 END) AS med_v_d_u_2020, SUM(CASE WHEN ano = 2020 AND modalidade = 'TOTAL' THEN media_viagens_diarias ELSE 0 END) AS med_v_d_t_2020,
SUM(CASE WHEN ano = 2021 AND modalidade = 'RODOVIARIO' THEN total_viagens_ano ELSE 0 END) AS tot_v_r_2021, SUM(CASE WHEN ano = 2021 AND modalidade = 'URBANO' THEN total_viagens_ano ELSE 0 END) AS tot_v_u_2021, SUM(CASE WHEN ano = 2021 AND modalidade = 'TOTAL' THEN total_viagens_ano ELSE 0 END) AS tot_v_t_2021, SUM(CASE WHEN ano = 2021 AND modalidade = 'RODOVIARIO' THEN media_viagens_diarias ELSE 0 END) AS med_v_d_r_2021, SUM(CASE WHEN ano = 2021 AND modalidade = 'URBANO' THEN media_viagens_diarias ELSE 0 END) AS med_v_d_u_2021, SUM(CASE WHEN ano = 2021 AND modalidade = 'TOTAL' THEN media_viagens_diarias ELSE 0 END) AS med_v_d_t_2021,
SUM(CASE WHEN ano = 2022 AND modalidade = 'RODOVIARIO' THEN total_viagens_ano ELSE 0 END) AS tot_v_r_2022, SUM(CASE WHEN ano = 2022 AND modalidade = 'URBANO' THEN total_viagens_ano ELSE 0 END) AS tot_v_u_2022, SUM(CASE WHEN ano = 2022 AND modalidade = 'TOTAL' THEN total_viagens_ano ELSE 0 END) AS tot_v_t_2022, SUM(CASE WHEN ano = 2022 AND modalidade = 'RODOVIARIO' THEN media_viagens_diarias ELSE 0 END) AS med_v_d_r_2022, SUM(CASE WHEN ano = 2022 AND modalidade = 'URBANO' THEN media_viagens_diarias ELSE 0 END) AS med_v_d_u_2022, SUM(CASE WHEN ano = 2022 AND modalidade = 'TOTAL' THEN media_viagens_diarias ELSE 0 END) AS med_v_d_t_2022,
SUM(CASE WHEN ano = 2023 AND modalidade = 'RODOVIARIO' THEN total_viagens_ano ELSE 0 END) AS tot_v_r_2023, SUM(CASE WHEN ano = 2023 AND modalidade = 'URBANO' THEN total_viagens_ano ELSE 0 END) AS tot_v_u_2023, SUM(CASE WHEN ano = 2023 AND modalidade = 'TOTAL' THEN total_viagens_ano ELSE 0 END) AS tot_v_t_2023, SUM(CASE WHEN ano = 2023 AND modalidade = 'RODOVIARIO' THEN media_viagens_diarias ELSE 0 END) AS med_v_d_r_2023, SUM(CASE WHEN ano = 2023 AND modalidade = 'URBANO' THEN media_viagens_diarias ELSE 0 END) AS med_v_d_u_2023, SUM(CASE WHEN ano = 2023 AND modalidade = 'TOTAL' THEN media_viagens_diarias ELSE 0 END) AS med_v_d_t_2023,
SUM(CASE WHEN ano = 2024 AND modalidade = 'RODOVIARIO' THEN total_viagens_ano ELSE 0 END) AS tot_v_r_2024, SUM(CASE WHEN ano = 2024 AND modalidade = 'URBANO' THEN total_viagens_ano ELSE 0 END) AS tot_v_u_2024, SUM(CASE WHEN ano = 2024 AND modalidade = 'TOTAL' THEN total_viagens_ano ELSE 0 END) AS tot_v_t_2024, SUM(CASE WHEN ano = 2024 AND modalidade = 'RODOVIARIO' THEN media_viagens_diarias ELSE 0 END) AS med_v_d_r_2024, SUM(CASE WHEN ano = 2024 AND modalidade = 'URBANO' THEN media_viagens_diarias ELSE 0 END) AS med_v_d_u_2024, SUM(CASE WHEN ano = 2024 AND modalidade = 'TOTAL' THEN media_viagens_diarias ELSE 0 END) AS med_v_d_t_2024
FROM tb_viagens_por_cidade_total_rodoviario_urbano
GROUP BY municipio
ORDER BY municipio ASC
""")

In [ ]:
# tb_viagens_por_cidade_total_rodoviario_urbano_pivot_tudo.show()

In [ ]:
tb_viagens_por_cidade_total_rodoviario_urbano_pivot_tudo.repartition(1).write.csv(f"./d_painel/{get_time()}_100_trp_sc_tb_viagens_por_cidade_total_rodoviario_urbano_pivot_tudo/", header = True, sep = ";", encoding = "utf-8")

In [ ]:
tb_viagens_por_cidade_total_rodoviario_urbano_pivot_modalidades = spark.sql("""
SELECT
ano
, municipio
, SUM(CASE WHEN modalidade = 'RODOVIARIO' THEN total_viagens_ano ELSE 0 END) AS tot_v_r
, SUM(CASE WHEN modalidade = 'URBANO' THEN total_viagens_ano ELSE 0 END) AS tot_v_u
, SUM(CASE WHEN modalidade = 'TOTAL' THEN total_viagens_ano ELSE 0 END) AS tot_v_t
, SUM(CASE WHEN modalidade = 'RODOVIARIO' THEN media_viagens_diarias ELSE 0 END) AS med_v_d_r
, SUM(CASE WHEN modalidade = 'URBANO' THEN media_viagens_diarias ELSE 0 END) AS med_v_d_u
, SUM(CASE WHEN modalidade = 'TOTAL' THEN media_viagens_diarias ELSE 0 END) AS med_v_d_t
FROM tb_viagens_por_cidade_total_rodoviario_urbano
GROUP BY ano, municipio
ORDER BY ano ASC, municipio ASC
""")

In [ ]:
tb_viagens_por_cidade_total_rodoviario_urbano_pivot_modalidades.repartition(1).write.csv(f"./d_painel/{get_time()}_100_trp_sc_tb_viagens_por_cidade_total_rodoviario_urbano_pivot_modalidades/", header = True, sep = ";", encoding = "utf-8")

### Tipologias de linhas

In [ ]:
tipologia_select = ""
tipologia = spark.sql(f"""
SELECT ano_prefixo_empresacod, prefixo, FIRST(tipologia_regiao) AS tipologia_regiao FROM (
SELECT
CONCAT(ano, '_', prefixo, '_', empresa_codigo) AS ano_prefixo_empresacod
,ano
,empresa_codigo
,empresa_alias
,prefixo
,prefixo_linha
,prefixo_ramal
,modalidade
,extensao
,secoes
,passageiros
,sig_linha_rgi_escala
,sig_linha_rgint_escala
,linha_sie
,linha_ibge
,CASE
    WHEN modalidade = 'URBANO' THEN 'Local'
    WHEN extensao <= 75.0 THEN 'Local'
    WHEN linha_ibge LIKE '%FLORIANÓPOLISSC%' THEN 'Troncal'
    WHEN sig_linha_rgint_escala = 'Inter' AND extensao BETWEEN 75.0 AND 250.0 THEN 'Inter-regional média'
    WHEN sig_linha_rgint_escala = 'Inter' AND extensao > 250.0 THEN 'Inter-regional longa'
    WHEN sig_linha_rgint_escala = 'Intra' THEN 'Regional'
    ELSE 'Não se aplica'
    END AS tipologia_regiao
    
FROM trp_sc_prefixos_1999_2023
--WHERE ano = 2024 AND passageiros > 0 AND passageiros IS NOT NULL
ORDER BY
CAST(ano AS INTEGER) ASC
,CAST(empresa_codigo AS INTEGER) ASC
,CAST(prefixo_linha AS INTEGER) ASC
,CAST(prefixo_ramal AS INTEGER) ASC
                ) WHERE tipologia_regiao LIKE '%{tipologia_select}%' GROUP BY ano_prefixo_empresacod, prefixo
""")

# tipologia.show(10, False)
tipologia.toPandas().to_csv("./d_painel/teste_dicionario_tipologias.csv", sep = ";", index = False, encoding = "utf-8")

### renomear 2

In [ ]:
# Renomeando e movendo os arquivos para a pasta d_painel raiz
renomearRaiz()